# Avito NLP: тестовое задание 
Камышанский Андрей, kamyshanskii.as@phystech.edu

# Общая идея решения

Задача: По запросу пользователя найти не более 50 наиболее релевантных предложений.

Главным архитектурным решением было разделение задачи на две последовательные подзадачи:

**поисковый запрос → поиск кандидатов → ранжирование кандидатов → 50 итоговых объявлений**

То есть итоговая система состоит из двух основных этапов:

1. **Candidate generation** — найти среди всего корпуса относительно небольшой набор объявлений, в котором находится как можно больше релевантных объектов.
2. **Ranking** — правильно упорядочить найденных кандидатов и выбрать из них 50 лучших.

Такое разделение позволяет улучшать итоговое решение, отдельно улучшая решение каждой из подзадач.



## Метрики

Основная метрика задачи — **Recall@50** = доля релевантных объявленийв итоговых 50 результатах.

Для оценки первого этапа используется **Candidate Recall** = доля релевантных объявлений, дошедших до этапа ранжирования.  

Поэтому Candidate Recall задаёт верхний предел качества ранжирования: если нужного объявления нет среди кандидатов, ranker уже не может добавить его в итоговый ответ.

Для отдельной оценки второго этапа используется **ranking efficiency** = **Recall@50 / Candidate Recall**

Она показывает, какую долю релевантных объявлений, найденных на первом этапе, ранжировщик добавил в итоговый ответ.



## Развитие решения

Сначала были построены базовые модели:

**Candidate generation** на основе эмбеддингов E5  
и  
**Ranking** на основе RRF.

Для такого baseline был получен **Recall@50 ≈ 0.153** на локальной validation-выборке.

После этого отдельно улучшался этап **Candidate generation**.

Были реализованы:

- **BM25** — поиск по лексическим совпадениям;
- разные представления текста объявления;
- передача дополнительных параметров запроса и объявления;
- объединение BM25 и E5;
- использование контекста запроса: категории, географии и других признаков.

Главной метрикой на этом этапе была **Candidate Recall**, так как она показывает, сколько релевантных объявлений вообще дошло до этапа ранжирования.  
При этом **Recall@50** использовался для оценки того, насколько изменения Candidate generation влияют на итоговый результат при неизменном Ranking.

Наибольший прирост на этом этапе дало добавление контекста запроса.  
Использование только глобального поиска по BM25 и E5 давало Recall@50 около **0.16**, тогда как добавление контекстного поиска позволило поднять его примерно до **0.74**.

После того как Candidate generation стал достаточно сильным, основной упор был сделан на улучшение **Ranking**.

Сначала для ранжирования кандидатов был обучен **CatBoost Ranker** на признаках, полученных на этапе Candidate generation:

- BM25 score;
- E5 similarity;
- позиции кандидата в разных retrieval-моделях;
- совпадение категории;
- совпадение географии;
- дополнительные признаки запроса и объявления.

При этом оказалось, что полный переход от RRF к CatBoost не всегда улучшает итоговый Recall@50. Поэтому был использован **гибридный подход**, объединяющий исходный ranking и CatBoost. Это позволило получить Recall@50 около **0.77**.

При этом оказалось, что полный переход от RRF к CatBoost не всегда улучшает итоговый Recall@50. Поэтому был использован **гибридный подход**, объединяющий исходный ranking и CatBoost. Это позволило получить Recall@50 около **0.77**.

Следующим шагом стало использование **Cross-Encoder**.

В E5 запрос и объявление кодируются независимо. Это позволяет быстро искать по всему корпусу, но ограничивает точность сравнения отдельных пар.  
Cross-Encoder получает запрос и объявление одновременно и может более точно оценивать их соответствие.

Применять Cross-Encoder ко всему корпусу было слишком дорого, поэтому он используется только для повторного ранжирования уже найденных кандидатов.

Сначала была проверена готовая BGE reranker модель без дополнительного обучения. При неизменном Candidate Recall она подняла Recall@50 примерно с **0.776 до 0.797**.  
После fine-tuning на наших данных Recall@50 вырос примерно до **0.812**.

Для обучения использовались **hard negatives** — объявления, которые предыдущий ranker считает похожими на запрос, что научило модель различать близкие объявления.

При сегментации групп ответов выяснилось, что на запросах без указанной географии модель ошибается значительно сильнее, чем при наличии георграфии, поэтому запросы теперь обрабатывались разными ветками:

- **regular** — запросы с географией;
- **special** — запросы, для которых требуется более широкий поиск.

В результате для них генерировались разное колличество кандидатов:

**regular → выдающая 150 кондидатов**  
**special → выдающая 300 кандидатов**

После этого проверялись более сложные варианты объединения моделей и динамического выбора числа кандидатов, но стабильного улучшения они не дали.

Итоговая архитектура решения:

**запрос → BM25 + E5 + context-aware Candidate generation → CatBoost Ranking → BGE Cross-Encoder reranking → объединение ranking-сигналов → Top-50**

## 1. Конфигурация эксперимента и окружение

Для обучения Cross-Encoder используется:

- `40 000` regular contexts;
- `25 000` special contexts;
- до `2` positive объявлений на один context;
- `3` hard negative на каждый positive;
- одна эпоха обучения.

Для финального ранжирования используется `150` кандидатов для regular запросов и `300` для special запросов.

In [1]:
from pathlib import Path
import os, json, hashlib

PROJECT_ROOT = Path("/home/ubuntu/avito")
V4_RUN_DIR = PROJECT_ROOT / "artifacts/context_ranker_v4/3ec8c4a5aac3b5d9"
V5_WORK_DIR = PROJECT_ROOT / "artifacts/final_v51_clean"

if not (V4_RUN_DIR / "run_completed.json").exists():
    raise FileNotFoundError(f"Не найдены подготовленные retrieval-артефакты: {V4_RUN_DIR}")

CFG = {
    "seed": 42,
    "fresh_holdout_seed": 20260927,

    # retrieval representation
    "dense_mode": "on",
    "max_seq_length": 256,
    "title_chars": 350,
    "params_chars": 650,
    "description_chars": 1000,

    # group-safe validation protocol
    "old_outer_fraction": 0.15,
    "old_inner_fraction": 0.15,
    "fresh_holdout_fraction": 0.15,
    "inner_valid_fraction": 0.15,

    # base ranker contexts
    "regular_fit_contexts": 12000,
    "regular_tune_contexts": 2500,
    "regular_refit_contexts": 20000,
    "regular_alpha": 0.40,
    "special_fit_contexts": 30000,
    "special_tune_contexts": 6000,
    "special_refit_contexts": 30000,
    "regular_global_depth": 300,
    "regular_context_depth": 100,
    "special_global_depth_grid": [300, 600, 1000],
    "special_proxy_depth": 250,
    "special_filter_depth": 300,
    "rrf_k": 60,
    "ranker_threads": min(8, os.cpu_count() or 2),
    "final_regular_fit_contexts": 20000,
    "final_special_fit_contexts": 30000,

    # cross-encoder training: selected configuration
    "train_regular_contexts": 40000,
    "train_special_contexts": 25000,
    "mine_regular_depth": 100,
    "mine_special_depth": 100,
    "max_pos_per_context": 2,
    "negatives_per_positive": 3,

    # cross-encoder
    "ce_model": "BAAI/bge-reranker-v2-m3",
    "ce_max_length": 384,
    "ce_train_batch_pairs": 4,
    "ce_grad_accum": 8,
    "ce_epochs": 1,
    "ce_learning_rate": 1e-4,
    "ce_weight_decay": 0.01,
    "ce_max_grad_norm": 1.0,
    "ce_warmup_ratio": 0.05,
    "ce_lora_r": 16,
    "ce_lora_alpha": 32,
    "ce_lora_dropout": 0.05,
    "ce_infer_batch": 32,

    # final adaptive inference
    "regular_K": 150,
    "special_K": 300,
    "gamma": 0.5,

    # runtime safety
    "checkpoint_optimizer_steps": 500,
    "min_free_disk_gb": 12,
    "holdout_contexts": 5000,
}

assert CFG["ce_epochs"] == 1
assert CFG["max_pos_per_context"] == 2
assert CFG["negatives_per_positive"] == 3


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


source_files = [
    V4_RUN_DIR / "prepared.json",
    V4_RUN_DIR / "documents.json",
    V4_RUN_DIR / "queries.json",
    V4_RUN_DIR / "v4_selection.json",
    V4_RUN_DIR / "regular_ranker_validation.cbm",
    V4_RUN_DIR / "regular_ranker_holdout.cbm",
    V4_RUN_DIR / "regular_ranker_submission.cbm",
]
missing = [str(p) for p in source_files if not p.exists()]
if missing:
    raise FileNotFoundError(f"Не хватает retrieval-артефактов: {missing}")

V5_SIGNATURE = {
    "pipeline": "final_v51_clean_v1",
    "cfg": CFG,
    "retrieval_run": V4_RUN_DIR.name,
    "source_sha256": {p.name: file_sha256(p) for p in source_files},
}
RUN_HASH = hashlib.sha256(json.dumps(V5_SIGNATURE, sort_keys=True).encode()).hexdigest()[:16]
V5_DIR = V5_WORK_DIR / f"run_{RUN_HASH}"
V5_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
import sys
import subprocess
import importlib
import gc
import re
import json
import math
import hashlib
import random
import unicodedata
from collections import OrderedDict
from functools import lru_cache

def ensure_package(module, requirement):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", requirement],
            check=True,
        )

for module, requirement in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("sklearn", "scikit-learn"),
    ("pyarrow", "pyarrow"),
    ("catboost", "catboost>=1.2.7,<2"),
    ("tqdm", "tqdm"),
    ("transformers", "transformers>=4.44,<5"),
    ("peft", "peft>=0.12,<1"),
    ("sentence_transformers", "sentence-transformers>=3,<6"),
]:
    ensure_package(module, requirement)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from catboost import CatBoostRanker
from torch.utils.data import Dataset, DataLoader
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    PeftModel,
)

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
torch.cuda.manual_seed_all(CFG["seed"])

if not torch.cuda.is_available():
    raise RuntimeError("v5 требует CUDA GPU.")

torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")


def atomic_json(path, obj):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(
        json.dumps(
            obj,
            ensure_ascii=False,
            indent=2,
            default=str,
        ),
        encoding="utf-8",
    )
    os.replace(tmp, path)


identity_path = V5_DIR / "v5_identity.json"
if identity_path.exists():
    old_identity = json.loads(
        identity_path.read_text()
    )
    if old_identity != V5_SIGNATURE:
        raise RuntimeError(
            "V5_DIR уже содержит другой config. "
            "Измените V5_RUN_ID перед новым экспериментом."
        )
else:
    atomic_json(
        identity_path,
        V5_SIGNATURE,
    )

/home/ubuntu/avito/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 2.0 Скачивание данных(для локального запуска раскомментировать)

In [ ]:
# # Скачать данные и подготовленные артефакты
# import requests
# import zipfile
#
# url = "ССЫЛКА_НА_ЯНДЕКС_ДИСК"
# api = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
# download_url = requests.get(api, params={"public_key": url}).json()["href"]
#
# archive = requests.get(download_url).content
# with open("data.zip", "wb") as f:
#     f.write(archive)
#
# with zipfile.ZipFile("data.zip") as z:
#     z.extractall("/home/ubuntu/avito")

## 3. Данные

Используются три исходные таблицы:

- `train.parquet` — обучающие взаимодействия;
- `benchmark_items.parquet` — объявления, среди которых нужно искать;
- `benchmark_queries.parquet` — запросы, для которых нужно построить итоговую выдачу.

В `train.parquet` один и тот же текст объявления может повторяться во многих взаимодействиях. Поэтому для дальнейшей работы данные переводятся в более удобное представление, где объявления, запросы и positive-пары хранятся отдельно.

In [3]:
import pyarrow as pa
import pyarrow.parquet as pq
from collections import Counter
from IPython.display import display

RAW_PATHS = {
    "train": PROJECT_ROOT / "data/raw/train.parquet",
    "benchmark_items": PROJECT_ROOT / "data/raw/benchmark_items.parquet",
    "benchmark_queries": PROJECT_ROOT / "data/raw/benchmark_queries.parquet",
}
missing_raw = [str(p) for p in RAW_PATHS.values() if not p.is_file()]
if missing_raw:
    raise FileNotFoundError(f"Missing raw parquet files: {missing_raw}")

raw_files = {name: pq.ParquetFile(path) for name, path in RAW_PATHS.items()}
raw_overview = pd.DataFrame([
    {
        "table": name,
        "rows": pf.metadata.num_rows,
        "columns": len(pf.schema_arrow.names),
    }
    for name, pf in raw_files.items()
])
display(raw_overview)

,table,rows,columns
0,train,497673,19
1,benchmark_items,189212,14
2,benchmark_queries,2452,6


### Подготовка обучающих пар

Одинаковая пара `context_id – item_id` хранится одним экземпляром, а число её повторений сохраняется в `pair_count`.

- `contexts` — уникальные поисковые контексты;
- `items` — уникальные объявления;
- `pairs` — известные релевантные пары `context – item`;
- `queries` — benchmark-запросы.

In [4]:
required = [
    "pairs.parquet",
    "contexts.parquet",
    "queries.parquet",
    "items.parquet",
    "prepared.json",
    "documents.npy",
    "queries.npy",
    "v4_selection.json",
    "regular_ranker_validation.cbm",
    "regular_ranker_holdout.cbm",
    "regular_ranker_submission.cbm",
]

missing = [
    name
    for name in required
    if not (V4_RUN_DIR / name).exists()
]
if missing:
    raise FileNotFoundError(
        f"Не хватает подготовленных retrieval-файлов: {missing}"
    )

pairs = pd.read_parquet(V4_RUN_DIR / "pairs.parquet")
contexts = pd.read_parquet(V4_RUN_DIR / "contexts.parquet")
queries = pd.read_parquet(V4_RUN_DIR / "queries.parquet")
items = pd.read_parquet(V4_RUN_DIR / "items.parquet")

BENCH_N = int(
    json.loads(
        (V4_RUN_DIR / "prepared.json").read_text()
    )["benchmark_count"]
)

selection = json.loads(
    (V4_RUN_DIR / "v4_selection.json").read_text()
)

BEST_SPECIAL_CONFIG = selection["special_retrieval"]
BEST_SPECIAL_ALPHA = float(selection["special_alpha"])
USE_SPECIAL_EXPERT = bool(selection["special_enabled"])

def special_config_token(config):
    return hashlib.sha256(
        json.dumps(
            config,
            sort_keys=True,
        ).encode()
    ).hexdigest()[:12]

SELECTED_SPECIAL_TOKEN = special_config_token(
    BEST_SPECIAL_CONFIG
)

item_index = pd.Index(items.item_id)
pairs["item_row"] = item_index.get_indexer(
    pairs.item_id
).astype("int32")

assert (pairs.item_row >= 0).all()
assert items.item_id.is_unique
assert queries.query_id.is_unique

positive_rows = (
    pairs.groupby("context_id", sort=False)
    .item_row
    .agg(lambda x: frozenset(map(int, x)))
    .to_dict()
)

context_index = contexts.set_index(
    "context_id",
    drop=False,
)

dataset_overview = pd.DataFrame({
    "object": ["items", "benchmark items", "positive pairs", "contexts", "benchmark queries"],
    "count": [len(items), BENCH_N, len(pairs), len(contexts), len(queries)],
})
display(dataset_overview)

,object,count
0,items,515895
1,benchmark items,189212
2,positive pairs,467029
3,contexts,354312
4,benchmark queries,2452


In [5]:
def first_column(columns, *names):
    lower = {c.casefold(): c for c in columns}
    return next((lower[n.casefold()] for n in names if n.casefold() in lower), None)

train_columns = raw_files["train"].schema_arrow.names
raw_rows = raw_files["train"].metadata.num_rows
pair_count = int(pairs.pair_count.sum())

eda_counts = pd.DataFrame({
    "metric": [
        "raw interactions",
        "deduplicated positive pairs",
        "repeated interactions",
        "unique contexts",
        "unique normalized queries",
        "unique train items",
        "benchmark queries",
        "benchmark items",
    ],
    "value": [
        raw_rows,
        len(pairs),
        raw_rows - len(pairs),
        contexts.context_id.nunique(),
        contexts.query_norm.nunique(),
        pairs.item_id.nunique(),
        raw_files["benchmark_queries"].metadata.num_rows,
        raw_files["benchmark_items"].metadata.num_rows,
    ],
})
display(eda_counts)

assert pairs.context_id.nunique() <= contexts.context_id.nunique()
assert len(pairs) <= raw_rows and pair_count >= len(pairs)
assert queries.query_id.is_unique and items.item_id.is_unique

,metric,value
0,raw interactions,497673
1,deduplicated positive pairs,467029
2,repeated interactions,30644
3,unique contexts,354312
4,unique normalized queries,73903
5,unique train items,344825
6,benchmark queries,2452
7,benchmark items,189212


### Представление текста

Для каждого объявления формируется текстовое представление из названия, параметров и описания.

In [6]:
def raw_lengths(table, labels):
    pf = raw_files[table]
    col = first_column(pf.schema_arrow.names, *labels)
    if col is None:
        return None, None
    all_lengths = []
    for batch in pf.iter_batches(batch_size=32768, columns=[col]):
        all_lengths.append(np.fromiter((len(str(x)) if x is not None else 0
                                       for x in batch.column(0).to_pylist()),
                                       dtype=np.int32, count=batch.num_rows))
    return col, np.concatenate(all_lengths)

def length_summary(values):
    a = np.asarray(values, dtype=np.int32)
    return {"mean": round(float(a.mean()), 1), "median": int(np.median(a)),
            "p90": int(np.percentile(a, 90)), "p95": int(np.percentile(a, 95)),
            "max": int(a.max())}

measurements = {}
raw_length_fields = {
    "raw query": ("train", ["query_text_raw", "query_raw", "search_query", "query", "query_text"]),
    "raw query params": ("benchmark_queries", ["query_infm_params_text", "query_params_text", "query_params", "params"]),
    "raw item title": ("benchmark_items", ["item_title_raw", "title"]),
    "raw item params": ("benchmark_items", ["item_infm_params_text", "item_params_text", "params"]),
    "raw item description": ("benchmark_items", ["item_description_raw", "description"]),
}
for label, (table, aliases) in raw_length_fields.items():
    source, lengths = raw_lengths(table, aliases)
    if source is not None:
        measurements[f"{label} [{table}.{source}]"] = length_summary(lengths)
    else:
        continue
    del lengths
measurements["final query_text [prepared contexts]"] = length_summary(contexts.query_text.str.len())
measurements["final item text [prepared items]"] = length_summary(items.text.str.len())
display(pd.DataFrame.from_dict(measurements, orient="index"))

raw_preview_col = first_column(raw_files["train"].schema_arrow.names,
                               "query_text_raw", "query_raw", "search_query", "query", "query_text")
if raw_preview_col:
    raw_examples = next(raw_files["train"].iter_batches(batch_size=1000, columns=[raw_preview_col])).column(0).to_pylist()
    prepared_text = contexts.drop_duplicates("query_norm").set_index("query_norm").query_text
    example_rows = []
    for value in raw_examples:
        if value is None: continue
        norm = unicodedata.normalize("NFKC", str(value)).casefold().replace("ё", "е")
        norm = " ".join(norm.split())
        if norm in prepared_text.index and norm not in [r["normalized query"] for r in example_rows]:
            example_rows.append({"raw query": str(value)[:160], "normalized query": norm,
                                 "query_text": str(prepared_text.loc[norm])[:240]})
        if len(example_rows) == 3: break
    display(pd.DataFrame(example_rows))
raw_item_cols = raw_files["benchmark_items"].schema_arrow.names
item_id_col = first_column(raw_item_cols, "item_id")
title_col = first_column(raw_item_cols, "item_title_raw", "title")
params_col = first_column(raw_item_cols, "item_infm_params_text", "item_params_text", "params")
description_col = first_column(raw_item_cols, "item_description_raw", "description")
if item_id_col and all((title_col, params_col, description_col)):
    sample_cols = list(dict.fromkeys([item_id_col, title_col, params_col, description_col]))
    raw_item_sample = next(raw_files["benchmark_items"].iter_batches(batch_size=5, columns=sample_cols)).to_pandas()
    item_examples = raw_item_sample.merge(items[["item_id", "text"]],
        left_on=item_id_col, right_on="item_id", how="left", validate="one_to_one")
    item_examples = item_examples[[title_col, params_col, description_col, "text"]].head(3)
    for column in item_examples: item_examples[column] = item_examples[column].astype(str).str.slice(0, 220)
    display(item_examples.rename(columns={"text": "final item text"}))
else:
    item_examples = items[["title", "params", "text"]].head(3).copy()
    for column in item_examples: item_examples[column] = item_examples[column].astype(str).str.slice(0, 220)
    display(item_examples.rename(columns={"text": "final item text"}))
del item_examples

,mean,median,p90,p95,max
raw query [train.search_query],17.5,17,28,31,113
raw item title [benchmark_items.item_title_raw],33.6,35,48,50,100
raw item params [benchmark_items.item_infm_params_text],973.9,738,1863,2475,19068
raw item description [benchmark_items.item_description_raw],1404.9,1054,3127,4205,8494
final query_text [prepared contexts],51.1,47,96,117,669
final item text [prepared items],1281.6,1383,1695,1699,1752


,raw query,normalized query,query_text
0,скупка телевизоров,скупка телевизоров,скупка телевизоров
1,автоподбор,автоподбор,автоподбор рейтинг пользователя 4 звезды и выше
2,баня на дровах,баня на дровах,баня на дровах онлайн-запись тип услуги спа-ус...


,item_title_raw,item_infm_params_text,item_description_raw,final item text
0,Ремонт/выкуп компьют. и ноутбуков с выездом на...,Вид услуги Компьютерная помощь Место оказания ...,"Выезд на дом в любое время, вплоть до 23:00\n\...",ремонт/выкуп компьют. и ноутбуков с выездом на...
1,Обучение ребенка чтению,"Вид услуги Обучение, курсы Место оказания услу...","Дорогие родители и маленькие книголюбы! \n\nЯ,...","обучение ребенка чтению вид услуги обучение, к..."
2,Афрокудри 5+,"Вид услуги Красота, здоровье Место оказания ус...",ВНИМАНИЕ Укладка !!!! 🥳\n\nАФРОкудри отличный ...,"афрокудри 5+ вид услуги красота, здоровье мест..."


## 4. Train / Validation / Holdout

Обычное случайное разбиение отдельных строк здесь может привести к утечке: одинаковый или почти одинаковый запрос может оказаться и в Train, и в Validation.

Поэтому разбиение выполняется группами по нормализованному тексту запроса `query_norm`, при этом одинаковый `query_norm` не может одновременно находиться в разных частях.

Получаются три части:

- **Train** — используется для обучения;
- **Validation** — используется для оценки выбранных решений;
- **Holdout** — дополнительная проверка того, что результат сохраняется на другой части данных.


In [7]:
def get_contexts(pair_frame):
    ids = pair_frame.context_id.drop_duplicates()
    return context_index.loc[ids].reset_index(drop=True)

def choose_contexts(frame, limit, seed):
    if limit is None or len(frame) <= limit:
        return frame.reset_index(drop=True)
    order = frame.assign(_u=np.random.default_rng(seed).random(len(frame))).sort_values('_u')
    order['_round'] = order.groupby('query_norm', sort=False).cumcount()
    return (
        order.sort_values(['_round', '_u'])
        .head(limit)
        .drop(columns=['_round', '_u'])
        .reset_index(drop=True)
    )

# Группы, исключённые из последующего Train, чтобы сохранить прежний протокол оценки.
old_outer = GroupShuffleSplit(
    n_splits=1,
    test_size=CFG['old_outer_fraction'],
    random_state=CFG['seed'],
)
old_train_idx, old_valid_idx = next(
    old_outer.split(pairs, groups=pairs.query_norm)
)
old_outer_train = pairs.iloc[old_train_idx]
old_outer_valid = pairs.iloc[old_valid_idx]

old_inner = GroupShuffleSplit(
    n_splits=1,
    test_size=CFG['old_inner_fraction'],
    random_state=CFG['seed'] + 1,
)
_, old_tune_idx = next(
    old_inner.split(old_outer_train, groups=old_outer_train.query_norm)
)
old_tune_pairs = old_outer_train.iloc[old_tune_idx]
old_tune_q = choose_contexts(
    get_contexts(old_tune_pairs),
    2500,
    CFG['seed'] + 1,
)

previously_evaluated = (
    set(old_outer_valid.query_norm)
    | set(old_tune_q.query_norm)
)

# Holdout формируется только из не пересекающихся query_norm.
eligible = pairs.loc[
    ~pairs.query_norm.isin(previously_evaluated)
].copy()

fresh_split = GroupShuffleSplit(
    n_splits=1,
    test_size=CFG['fresh_holdout_fraction'],
    random_state=CFG['fresh_holdout_seed'],
)
_, fresh_idx = next(
    fresh_split.split(eligible, groups=eligible.query_norm)
)
fresh_groups = set(eligible.iloc[fresh_idx].query_norm)

holdout_pairs = pairs.loc[
    pairs.query_norm.isin(fresh_groups)
].copy()
dev_pairs = pairs.loc[
    ~pairs.query_norm.isin(fresh_groups)
].copy()

assert not (set(dev_pairs.query_norm) & set(holdout_pairs.query_norm))
assert not (set(holdout_pairs.query_norm) & previously_evaluated)

# Train/Validation внутри development части.
inner = GroupShuffleSplit(
    n_splits=1,
    test_size=CFG['inner_valid_fraction'],
    random_state=CFG['fresh_holdout_seed'] + 1,
)
fit_idx, tune_idx = next(
    inner.split(dev_pairs, groups=dev_pairs.query_norm)
)
fit_pairs = dev_pairs.iloc[fit_idx].copy()
tune_pairs = dev_pairs.iloc[tune_idx].copy()

assert not (set(fit_pairs.query_norm) & set(tune_pairs.query_norm))

fit_q = choose_contexts(
    get_contexts(fit_pairs),
    CFG['regular_fit_contexts'],
    CFG['seed'],
)
tune_q = choose_contexts(
    get_contexts(tune_pairs),
    CFG['regular_tune_contexts'],
    CFG['seed'] + 1,
)
holdout_q = get_contexts(holdout_pairs)

dev_refit_q = choose_contexts(
    get_contexts(dev_pairs),
    CFG['regular_refit_contexts'],
    CFG['seed'] + 20,
)

final_regular_q = choose_contexts(
    contexts,
    CFG['final_regular_fit_contexts'],
    CFG['seed'] + 30,
)

FULL_LOCATION_SET = set(
    items.loc[items.location >= 0, 'location'].astype(int)
)
BENCH_LOCATION_SET = set(
    items.iloc[:BENCH_N]
    .loc[lambda x: x.location >= 0, 'location']
    .astype(int)
)

def special_contexts(frame, location_set):
    return frame.loc[
        ~frame.location.isin(location_set)
    ].reset_index(drop=True)

special_fit_q = choose_contexts(
    special_contexts(get_contexts(fit_pairs), FULL_LOCATION_SET),
    CFG['special_fit_contexts'],
    CFG['seed'] + 40,
)
special_tune_q = choose_contexts(
    special_contexts(get_contexts(tune_pairs), FULL_LOCATION_SET),
    CFG['special_tune_contexts'],
    CFG['seed'] + 41,
)
special_dev_q = choose_contexts(
    special_contexts(get_contexts(dev_pairs), FULL_LOCATION_SET),
    CFG['special_refit_contexts'],
    CFG['seed'] + 42,
)

# Для benchmark exact-location определяется относительно benchmark corpus.
special_final_q = choose_contexts(
    special_contexts(contexts, BENCH_LOCATION_SET),
    CFG['final_special_fit_contexts'],
    CFG['seed'] + 43,
)

for name, frame in [
    ("train_reference", fit_q),
    ("validation", tune_q),
    ("holdout", holdout_q),
]:
    frame[["context_id", "query_norm"]].to_parquet(
        V5_DIR / f"{name}_contexts.parquet", index=False
    )

In [8]:
fit_groups = set(fit_pairs.query_norm)
inner_groups = set(tune_pairs.query_norm)
holdout_groups = set(holdout_pairs.query_norm)
assert not fit_groups & inner_groups
assert not fit_groups & holdout_groups
assert not inner_groups & holdout_groups

split_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Holdout"],
    "contexts": [len(get_contexts(fit_pairs)), len(get_contexts(tune_pairs)), len(get_contexts(holdout_pairs))],
    "query_norm": [len(fit_groups), len(inner_groups), len(holdout_groups)],
})
display(split_summary)

del raw_files
gc.collect()

,split,contexts,query_norm
0,Train,265215,55126
1,Validation,46861,9729
2,Holdout,42236,9048


17

## 5. Candidate generation

Первая подзадача — найти небольшой набор кандидатов, содержащий как можно больше релевантных объявлений.

Для этого используются несколько источников информации.

**BM25** отвечает за лексическое совпадение: одинаковые слова, числа, названия моделей и характеристики.

**E5** отвечает за семантическую близость и позволяет находить похожие объявления даже при отсутствии точного совпадения слов.

Дополнительно используется **контекст запроса**: категория, география и параметры.

Кандидаты из разных источников объединяются, после чего для них рассчитываются признаки и строится базовый ranking.

In [9]:
TOKEN_RE = re.compile(r'(?u)\b\w+\b')

def tokens(text):
    return TOKEN_RE.findall(str(text).casefold().replace('ё','е'))

class BM25:
    def fit(self, texts):
        self.vectorizer = CountVectorizer(tokenizer=tokens, token_pattern=None, lowercase=False,
                                          dtype=np.float32)
        # Фиктивный токен позволяет индексировать даже полностью пустое поле.
        matrix = self.vectorizer.fit_transform((str(x) if str(x).strip() else '__empty_field__' for x in texts)).tocsr()
        n = matrix.shape[0]
        lengths = np.asarray(matrix.sum(axis=1)).ravel()
        avg = max(float(lengths.mean()), 1e-6)
        df = np.bincount(matrix.indices, minlength=matrix.shape[1]).astype('float32')
        idf = np.log1p((n-df+0.5)/(df+0.5))
        for start in range(0, n, 2048):
            end = min(n, start+2048); a,b = matrix.indptr[start], matrix.indptr[end]
            row = np.repeat(np.arange(start,end), np.diff(matrix.indptr[start:end+1]))
            tf = matrix.data[a:b]
            matrix.data[a:b] = idf[matrix.indices[a:b]] * tf*2.5/(tf+1.5*(0.25+0.75*lengths[row]/avg))
        self.inverted = matrix.T.tocsr()
        self.n = n
        return self

    def score(self, text):
        query = self.vectorizer.transform([text])
        return (query @ self.inverted).toarray().ravel().astype('float32', copy=False)

bm_title = BM25().fit(items.title)
bm_text = BM25().fit(items.text)
bm_params = BM25().fit(items.params)

In [10]:
needed_q = pd.concat(
    [
        fit_q,
        tune_q,
        holdout_q,
        dev_refit_q,
        special_fit_q,
        special_tune_q,
        special_dev_q,
        final_regular_q,
        special_final_q,
        queries,
    ],
    ignore_index=True,
)

unique_query_texts = (
    needed_q.query_text
    .drop_duplicates()
    .tolist()
)
query_text_to_row = {
    text: i
    for i, text in enumerate(unique_query_texts)
}
del needed_q


def ordered_text_hash(texts):
    h = hashlib.sha256()
    for text in texts:
        b = str(text).encode("utf-8")
        h.update(
            len(b).to_bytes(
                8,
                "little",
            )
        )
        h.update(b)
    return h.hexdigest()

query_meta_path = V4_RUN_DIR / "queries.json"
document_meta_path = V4_RUN_DIR / "documents.json"

if query_meta_path.exists():
    query_meta = json.loads(
        query_meta_path.read_text()
    )
    expected_hash = (
        query_meta
        .get("identity", {})
        .get("text_hash")
    )
    actual_hash = ordered_text_hash(
        unique_query_texts
    )
    if (
        expected_hash is not None
        and expected_hash != actual_hash
    ):
        raise RuntimeError(
            "Порядок query_text не совпал с cached queries.npy. "
            "Нельзя безопасно использовать cached E5."
        )

if document_meta_path.exists():
    document_meta = json.loads(
        document_meta_path.read_text()
    )
    expected_hash = (
        document_meta
        .get("identity", {})
        .get("text_hash")
    )
    actual_hash = ordered_text_hash(
        items.text.astype(str).tolist()
    )
    if (
        expected_hash is not None
        and expected_hash != actual_hash
    ):
        raise RuntimeError(
            "items.text не совпадает с cached documents.npy."
        )


document_vectors = np.load(
    V4_RUN_DIR / "documents.npy",
    mmap_mode="r",
)
query_vectors = np.load(
    V4_RUN_DIR / "queries.npy",
    mmap_mode="r",
)

assert document_vectors.shape[0] == len(items)
assert query_vectors.shape[0] == len(unique_query_texts)

USE_DENSE = True
GPU_DOCUMENTS = None

def load_gpu_documents():
    global GPU_DOCUMENTS

    if GPU_DOCUMENTS is not None:
        return

    GPU_DOCUMENTS = torch.empty(
        document_vectors.shape,
        dtype=torch.float32,
        device="cuda",
    )

    for start in tqdm(
        range(0, len(items), 32768),
        desc="E5 documents -> GPU",
    ):
        end = min(start + 32768, len(items))
        GPU_DOCUMENTS[start:end] = torch.from_numpy(
            np.array(
                document_vectors[start:end],
                copy=True,
            )
        ).to("cuda")

def free_gpu_documents():
    global GPU_DOCUMENTS
    if GPU_DOCUMENTS is not None:
        del GPU_DOCUMENTS
        GPU_DOCUMENTS = None
    gc.collect()
    torch.cuda.empty_cache()

load_gpu_documents()

E5 documents -> GPU: 100%|██████████| 16/16 [00:00<00:00, 23.47it/s]


In [11]:
location_rows = {
    int(k): np.asarray(v, dtype="int32")
    for k, v in items.groupby(
        "location",
        sort=False,
    ).indices.items()
}

category_rows = {
    int(k): np.asarray(v, dtype="int32")
    for k, v in items.groupby(
        "category",
        sort=False,
    ).indices.items()
}

benchmark_location_rows = {}
for loc, rows in location_rows.items():
    b = rows[rows < BENCH_N]
    if len(b):
        benchmark_location_rows[int(loc)] = b

centres_frame = items.loc[
    items.latitude.between(-90, 90)
    & items.longitude.between(-180, 180)
]

centres = (
    centres_frame
    .groupby("location")[["latitude", "longitude"]]
    .median()
    .to_dict("index")
)

ITEM_LOCATIONS = items.location.to_numpy(
    dtype="int64"
)

def proxy_lookup_from_table(table):
    lookup = {}

    for search_loc, g in table.groupby(
        "search_location",
        sort=False,
    ):
        g = g.sort_values(
            ["rank", "item_location"]
        )

        locations = g.item_location.to_numpy(
            dtype="int64"
        )
        probs = g.prob.to_numpy(
            dtype="float32"
        )
        ranks = g["rank"].to_numpy(
            dtype="int32"
        )

        lookup[int(search_loc)] = {
            "locations": locations,
            "probs": probs,
            "ranks": ranks,
            "by_location": {
                int(loc): (
                    int(rank),
                    float(prob),
                )
                for loc, rank, prob
                in zip(
                    locations,
                    ranks,
                    probs,
                )
            },
        }

    return lookup

def proxy_token(table):
    values = pd.util.hash_pandas_object(
        table[
            [
                "search_location",
                "item_location",
                "rank",
                "prob",
            ]
        ],
        index=False,
    ).values

    return hashlib.sha256(
        values.tobytes()
    ).hexdigest()[:16]

proxy_tables = {
    name: pd.read_parquet(
        V4_RUN_DIR / f"proxy_{name}.parquet"
    )
    for name in ["fit", "dev", "full"]
}

proxy_lookups = {
    name: proxy_lookup_from_table(table)
    for name, table in proxy_tables.items()
}

proxy_tokens = {
    name: proxy_token(table)
    for name, table in proxy_tables.items()
}

In [12]:
def top_indices(scores, k, allowed=None, positive_only=False):
    ids = (
        np.arange(len(scores), dtype='int32')
        if allowed is None
        else np.asarray(allowed, dtype='int32')
    )
    values = scores[ids]
    mask = np.isfinite(values)

    if positive_only:
        mask &= values > 0

    ids, values = ids[mask], values[mask]
    k = min(int(k), len(ids))

    if not k:
        return np.empty(0, dtype='int32')

    if len(ids) > k:
        boundary = np.partition(
            values,
            len(values) - k,
        )[len(values) - k]

        above = ids[values > boundary]
        tied = np.sort(
            ids[values == boundary]
        )[:k - len(above)]

        ids = np.concatenate([above, tied])

    return ids[
        np.lexsort((ids, -scores[ids]))
    ].astype('int32')

REGULAR_CHANNELS = [
    'title',
    'text',
    'dense',
    'loc_lex',
    'loc_dense',
    'cat_lex',
    'cat_dense',
    'both_lex',
    'both_dense',
    'near_lex',
    'near_dense',
]

SPECIAL_CHANNELS = [
    'title',
    'text',
    'dense',
    'filters',
    'proxy_lex',
    'proxy_dense',
    'proxy_filters',
]

@lru_cache(maxsize=2048)
def nearest_locations(location):
    if location not in centres:
        return []

    keys = np.array(
        [
            k for k in centres
            if k >= 0 and k != location
        ],
        dtype='int64',
    )

    if not len(keys):
        return []

    lat = np.radians([
        centres[int(k)]['latitude']
        for k in keys
    ])
    lon = np.radians([
        centres[int(k)]['longitude']
        for k in keys
    ])

    a, b = np.radians([
        centres[location]['latitude'],
        centres[location]['longitude'],
    ])

    h = (
        np.sin((lat - a) / 2) ** 2
        + np.cos(lat)
        * np.cos(a)
        * np.sin((lon - b) / 2) ** 2
    )
    distance = 2 * 6371 * np.arcsin(
        np.sqrt(np.clip(h, 0, 1))
    )
    order = np.argsort(distance)

    return keys[
        order[distance[order] <= 100][:8]
    ].tolist()


class RetrievalV4:
    def __init__(self):
        self.cache = OrderedDict()
        self.scope = 'full'
        self.active = np.arange(len(items), dtype='int32')
        self.proxy_lookup = {}
        self.proxy_token = 'none'

    def set_scope(self, name):
        assert name in {'full', 'benchmark'}
        self.scope = name
        self.active = np.arange(
            BENCH_N if name == 'benchmark' else len(items),
            dtype='int32',
        )

    def set_proxy_lookup(self, lookup, token):
        if token != self.proxy_token:
            self.cache.clear()
        self.proxy_lookup = lookup
        self.proxy_token = token

    def _location_subset(self, location):
        location = int(location)

        if self.scope == 'benchmark':
            return benchmark_location_rows.get(
                location,
                np.empty(0, dtype='int32'),
            )

        return location_rows.get(
            location,
            np.empty(0, dtype='int32'),
        )

    def _category_subset(self, category):
        rows = category_rows.get(
            int(category),
            np.empty(0, dtype='int32'),
        )

        if self.scope == 'benchmark':
            rows = rows[rows < BENCH_N]

        return rows

    def has_exact_location(self, q):
        if int(q.location) < 0:
            return False
        return len(self._location_subset(int(q.location))) > 0

    def scores(self, q):
        key = (
            q.query_norm,
            q.params[:CFG['params_chars']],
        )

        if key in self.cache:
            self.cache.move_to_end(key)
            return self.cache[key]

        title = bm_title.score(q.query_norm)
        body = bm_text.score(q.query_norm)
        filters = bm_params.score(
            q.params[:CFG['params_chars']]
        )

        lexical = (
            title / max(float(title.max()), 1e-6)
            + body / max(float(body.max()), 1e-6)
        )

        if q.params:
            lexical += (
                0.5
                * filters
                / max(float(filters.max()), 1e-6)
            )

        dense = np.zeros(
            len(items),
            dtype='float32',
        )

        if USE_DENSE:
            with torch.inference_mode():
                vec = torch.from_numpy(
                    np.array(
                        query_vectors[
                            query_text_to_row[q.query_text]
                        ]
                    )
                ).to('cuda')

                dense = torch.mv(
                    GPU_DOCUMENTS,
                    vec,
                ).cpu().numpy()

        result = {
            'title': title,
            'text': body,
            'filters': filters,
            'lexical': lexical,
            'dense': dense,
        }

        self.cache[key] = result

        while len(self.cache) > 2:
            self.cache.popitem(last=False)

        return result

    def _cached_global_top(
        self,
        scores,
        cache_name,
        depth,
        positive_only=False,
    ):
        root = scores.setdefault(
            '_global_top',
            {},
        ).setdefault(
            self.scope,
            {},
        )

        entry = root.get(cache_name)

        if (
            entry is None
            or entry['depth'] < depth
        ):
            source = scores[cache_name]
            ids = top_indices(
                source,
                depth,
                self.active,
                positive_only=positive_only,
            )

            root[cache_name] = {
                'depth': int(depth),
                'ids': ids,
            }

            entry = root[cache_name]

        return entry['ids'][:depth]

    def retrieve_regular(self, q):
        s = self.scores(q)

        r = {
            name: np.empty(0, dtype='int32')
            for name in REGULAR_CHANNELS
        }

        r['title'] = self._cached_global_top(
            s,
            'title',
            CFG['regular_global_depth'],
            True,
        )
        # В v3 канал text ранжировался по combined lexical score.
        lexical_cache = s.setdefault(
            '_regular_lexical',
            {},
        )

        if self.scope not in lexical_cache:
            lexical_cache[self.scope] = top_indices(
                s['lexical'],
                CFG['regular_global_depth'],
                self.active,
                True,
            )

        r['text'] = lexical_cache[self.scope]

        if USE_DENSE:
            r['dense'] = self._cached_global_top(
                s,
                'dense',
                CFG['regular_global_depth'],
                False,
            )

        loc = (
            self._location_subset(q.location)
            if q.location >= 0
            else np.empty(0, dtype='int32')
        )

        cat = (
            self._category_subset(q.category)
            if q.category > 0
            else np.empty(0, dtype='int32')
        )

        both = np.intersect1d(
            loc,
            cat,
            assume_unique=True,
        )

        scopes = [('loc', loc)]

        if len(cat) < 0.95 * len(self.active):
            scopes.append(('cat', cat))

        if len(both) and not np.array_equal(both, loc):
            scopes.append(('both', both))

        if len(loc) < 50:
            near_parts = []

            for k in nearest_locations(int(q.location)):
                rows = self._location_subset(k)
                if len(rows):
                    near_parts.append(rows)

            if near_parts:
                near = np.unique(
                    np.concatenate(near_parts)
                )
                scopes.append(('near', near))

        for name, subset in scopes:
            if (
                not len(subset)
                or len(subset) == len(self.active)
            ):
                continue

            r[name + '_lex'] = top_indices(
                s['lexical'],
                CFG['regular_context_depth'],
                subset,
                True,
            )

            if USE_DENSE:
                r[name + '_dense'] = top_indices(
                    s['dense'],
                    CFG['regular_context_depth'],
                    subset,
                    False,
                )

        candidates = np.unique(
            np.concatenate(list(r.values()))
        ).astype('int32')

        if len(candidates) < min(50, len(self.active)):
            fallback = (
                np.nan_to_num(
                    items.rating.to_numpy(),
                    nan=0,
                ) * 0.05
                + np.log1p(
                    np.nan_to_num(
                        items.reviews.to_numpy(),
                        nan=0,
                    ).clip(0)
                )
            )

            candidates = np.union1d(
                candidates,
                top_indices(
                    fallback,
                    50,
                    self.active,
                ),
            ).astype('int32')

        return candidates, r, s

    def _proxy_subset(self, search_location, topn):
        if topn <= 0:
            return np.empty(0, dtype='int32')

        entry = self.proxy_lookup.get(
            int(search_location)
        )

        if entry is None:
            return np.empty(0, dtype='int32')

        locs = entry['locations'][:topn]

        parts = []

        for loc in locs:
            rows = self._location_subset(int(loc))
            if len(rows):
                parts.append(rows)

        if not parts:
            return np.empty(0, dtype='int32')

        return np.unique(
            np.concatenate(parts)
        ).astype('int32')

    def retrieve_special(
        self,
        q,
        global_depth,
        proxy_topn,
        use_filter,
    ):
        s = self.scores(q)

        r = {
            name: np.empty(0, dtype='int32')
            for name in SPECIAL_CHANNELS
        }

        r['title'] = self._cached_global_top(
            s,
            'title',
            global_depth,
            True,
        )

        special_root = s.setdefault(
            '_special_global',
            {},
        ).setdefault(
            self.scope,
            {},
        )

        lex_entry = special_root.get('lexical')

        if (
            lex_entry is None
            or lex_entry['depth'] < global_depth
        ):
            special_root['lexical'] = {
                'depth': int(global_depth),
                'ids': top_indices(
                    s['lexical'],
                    global_depth,
                    self.active,
                    True,
                ),
            }

        r['text'] = special_root[
            'lexical'
        ]['ids'][:global_depth]

        if USE_DENSE:
            r['dense'] = self._cached_global_top(
                s,
                'dense',
                global_depth,
                False,
            )

        if use_filter and q.params:
            r['filters'] = self._cached_global_top(
                s,
                'filters',
                CFG['special_filter_depth'],
                True,
            )

        proxy_cache = s.setdefault(
            '_proxy_channels',
            {},
        )

        proxy_key = (
            self.scope,
            self.proxy_token,
            int(q.location),
            int(proxy_topn),
            bool(use_filter),
        )

        if proxy_key not in proxy_cache:
            subset = self._proxy_subset(
                q.location,
                proxy_topn,
            )

            entry = {
                'proxy_lex': np.empty(
                    0,
                    dtype='int32',
                ),
                'proxy_dense': np.empty(
                    0,
                    dtype='int32',
                ),
                'proxy_filters': np.empty(
                    0,
                    dtype='int32',
                ),
            }

            if (
                len(subset)
                and len(subset) != len(self.active)
            ):
                entry['proxy_lex'] = top_indices(
                    s['lexical'],
                    CFG['special_proxy_depth'],
                    subset,
                    True,
                )

                if USE_DENSE:
                    entry['proxy_dense'] = top_indices(
                        s['dense'],
                        CFG['special_proxy_depth'],
                        subset,
                        False,
                    )

                if use_filter and q.params:
                    entry['proxy_filters'] = top_indices(
                        s['filters'],
                        CFG['special_proxy_depth'],
                        subset,
                        True,
                    )

            proxy_cache[proxy_key] = entry

        r.update(proxy_cache[proxy_key])

        candidates = np.unique(
            np.concatenate(list(r.values()))
        ).astype('int32')

        if len(candidates) < min(50, len(self.active)):
            fallback = (
                np.nan_to_num(
                    items.rating.to_numpy(),
                    nan=0,
                ) * 0.05
                + np.log1p(
                    np.nan_to_num(
                        items.reviews.to_numpy(),
                        nan=0,
                    ).clip(0)
                )
            )

            candidates = np.union1d(
                candidates,
                top_indices(
                    fallback,
                    50,
                    self.active,
                ),
            ).astype('int32')

        return candidates, r, s


retrieval = RetrievalV4()

In [13]:
@lru_cache(maxsize=32000)
def title_tokens(row):
    return frozenset(tokens(items.title.iat[row]))

@lru_cache(maxsize=12000)
def parameter_tokens(row):
    return frozenset(tokens(items.params.iat[row]))

def reciprocal_features(candidates, rankings, channel_names):
    c = np.asarray(candidates, dtype='int32')
    rr = np.zeros(
        (len(c), len(channel_names)),
        dtype='float32',
    )

    for j, name in enumerate(channel_names):
        ids = rankings.get(
            name,
            np.empty(0, dtype='int32'),
        )

        if not len(ids):
            continue

        pos = np.searchsorted(c, ids)
        inside = pos < len(c)
        ok = np.flatnonzero(inside)
        ok = ok[c[pos[ok]] == ids[ok]]

        rr[pos[ok], j] = (
            1
            / (
                CFG['rrf_k']
                + ok
                + 1
            )
        )

    return rr

def coverage_features(q, c):
    qt = frozenset(tokens(q.query_norm))
    ft = frozenset(
        tokens(
            q.params[:CFG['params_chars']]
        )
    )
    digits = frozenset(
        t for t in qt
        if any(ch.isdigit() for ch in t)
    )

    n = len(c)

    title_cover = np.fromiter(
        (
            len(qt & title_tokens(int(i)))
            / max(len(qt), 1)
            for i in c
        ),
        dtype='float32',
        count=n,
    )

    filter_cover = np.fromiter(
        (
            len(ft & parameter_tokens(int(i)))
            / max(len(ft), 1)
            if ft else 0
            for i in c
        ),
        dtype='float32',
        count=n,
    )

    digit_cover = np.fromiter(
        (
            len(digits & title_tokens(int(i)))
            / max(len(digits), 1)
            if digits else 0
            for i in c
        ),
        dtype='float32',
        count=n,
    )

    return (
        qt,
        ft,
        digits,
        title_cover,
        filter_cover,
        digit_cover,
    )

def regular_features(q, candidates, rankings, scores):
    c = np.asarray(candidates, dtype='int32')
    docs = items.iloc[c]
    n = len(c)

    rr = reciprocal_features(
        c,
        rankings,
        REGULAR_CHANNELS,
    )

    (
        qt,
        ft,
        digits,
        title_cover,
        filter_cover,
        digit_cover,
    ) = coverage_features(q, c)

    location_known = (
        (q.location >= 0)
        & (docs.location.to_numpy() >= 0)
    )
    category_known = (
        (q.category > 0)
        & (docs.category.to_numpy() > 0)
    )
    same_location = (
        location_known
        & (docs.location.to_numpy() == q.location)
    )
    same_category = (
        category_known
        & (docs.category.to_numpy() == q.category)
    )

    distance = np.full(
        n,
        np.nan,
        dtype='float64',
    )

    if int(q.location) in centres and q.location >= 0:
        centre = centres[int(q.location)]
        lat = np.radians(
            docs.latitude.to_numpy()
        )
        lon = np.radians(
            docs.longitude.to_numpy()
        )
        lat0, lon0 = np.radians([
            centre['latitude'],
            centre['longitude'],
        ])

        a = (
            np.sin((lat - lat0) / 2) ** 2
            + np.cos(lat)
            * np.cos(lat0)
            * np.sin((lon - lon0) / 2) ** 2
        )
        distance = (
            2
            * 6371
            * np.arcsin(
                np.sqrt(np.clip(a, 0, 1))
            )
        )

        invalid = (
            ~docs.latitude.between(-90, 90).to_numpy()
            | ~docs.longitude.between(-180, 180).to_numpy()
        )
        distance[invalid] = np.nan

    price = docs.price.to_numpy()
    rating = docs.rating.to_numpy()
    reviews = docs.reviews.to_numpy()

    rating_filter = re.search(
        r'рейтинг.{0,40}?([1-5])\s+зв',
        q.params,
    )
    min_rating = (
        float(rating_filter.group(1))
        if rating_filter
        else 0
    )

    data = OrderedDict([
        ('bm25_title', scores['title'][c]),
        ('bm25_text', scores['text'][c]),
        ('bm25_filters', scores['filters'][c]),
        ('lexical', scores['lexical'][c]),
        ('cosine', scores['dense'][c]),
        ('dense_enabled', np.full(n, float(USE_DENSE))),
        ('title_coverage', title_cover),
        ('filter_coverage', filter_cover),
        ('digit_coverage', digit_cover),
        ('query_words', np.full(n, len(qt))),
        ('filter_words', np.full(n, len(ft))),
        ('has_digits', np.full(n, bool(digits))),
        ('same_location', same_location),
        ('location_known', location_known),
        ('same_category', same_category),
        ('category_known', category_known),
        ('delivery_search', np.full(n, q.delivery)),
        ('distance_log_km', np.log1p(distance)),
        ('distance_known', np.isfinite(distance)),
        ('price_log', np.log1p(np.maximum(price, 0))),
        ('price_known', np.isfinite(price)),
        ('rating', rating),
        ('rating_known', np.isfinite(rating)),
        ('reviews_log', np.log1p(np.maximum(reviews, 0))),
        ('reviews_known', np.isfinite(reviews)),
        ('rating_filter', np.full(n, min_rating)),
        (
            'rating_meets_filter',
            (rating >= min_rating) & (min_rating > 0),
        ),
        ('phone_hidden', docs.phone_hidden.to_numpy()),
        ('message_forbidden', docs.message_forbidden.to_numpy()),
        ('title_chars', docs.title.str.len().to_numpy()),
        ('params_chars', docs.params.str.len().to_numpy()),
        ('rrf_global', rr[:, :3].sum(axis=1)),
        ('rrf_context', rr.sum(axis=1)),
        ('channel_count', (rr > 0).sum(axis=1)),
    ])

    data.update(
        (
            f'rr_{name}',
            rr[:, j],
        )
        for j, name in enumerate(REGULAR_CHANNELS)
    )

    out = np.column_stack(
        list(data.values())
    ).astype('float32')
    out[~np.isfinite(out)] = -1.0

    return out, list(data)

def proxy_candidate_features(q, candidates, lookup):
    entry = lookup.get(int(q.location))

    ranks = np.zeros(
        len(candidates),
        dtype='int32',
    )
    probs = np.zeros(
        len(candidates),
        dtype='float32',
    )

    if entry is None:
        return ranks, probs

    mapping = entry['by_location']
    locs = ITEM_LOCATIONS[
        np.asarray(candidates, dtype='int32')
    ]

    for j, loc in enumerate(locs):
        value = mapping.get(int(loc))
        if value is not None:
            ranks[j] = value[0]
            probs[j] = value[1]

    return ranks, probs

def special_features(
    q,
    candidates,
    rankings,
    scores,
    proxy_lookup,
):
    c = np.asarray(candidates, dtype='int32')
    docs = items.iloc[c]
    n = len(c)

    rr = reciprocal_features(
        c,
        rankings,
        SPECIAL_CHANNELS,
    )

    (
        qt,
        ft,
        digits,
        title_cover,
        filter_cover,
        digit_cover,
    ) = coverage_features(q, c)

    proxy_rank, proxy_prob = (
        proxy_candidate_features(
            q,
            c,
            proxy_lookup,
        )
    )

    price = docs.price.to_numpy()
    rating = docs.rating.to_numpy()
    reviews = docs.reviews.to_numpy()

    same_category = (
        (q.category > 0)
        & (docs.category.to_numpy() > 0)
        & (docs.category.to_numpy() == q.category)
    )

    data = OrderedDict([
        ('bm25_title', scores['title'][c]),
        ('bm25_text', scores['text'][c]),
        ('bm25_filters', scores['filters'][c]),
        ('lexical', scores['lexical'][c]),
        ('cosine', scores['dense'][c]),
        ('title_coverage', title_cover),
        ('filter_coverage', filter_cover),
        ('digit_coverage', digit_cover),
        ('query_words', np.full(n, len(qt))),
        ('filter_words', np.full(n, len(ft))),
        ('has_digits', np.full(n, bool(digits))),
        ('same_category', same_category),
        ('price_log', np.log1p(np.maximum(price, 0))),
        ('price_known', np.isfinite(price)),
        ('rating', rating),
        ('rating_known', np.isfinite(rating)),
        ('reviews_log', np.log1p(np.maximum(reviews, 0))),
        ('reviews_known', np.isfinite(reviews)),
        ('phone_hidden', docs.phone_hidden.to_numpy()),
        ('message_forbidden', docs.message_forbidden.to_numpy()),
        ('title_chars', docs.title.str.len().to_numpy()),
        ('params_chars', docs.params.str.len().to_numpy()),

        # Proxy geography
        (
            'proxy_rank_recip',
            np.divide(
                1.0,
                proxy_rank,
                out=np.zeros(
                    len(proxy_rank),
                    dtype='float32',
                ),
                where=proxy_rank > 0,
            ),
        ),
        ('proxy_probability', proxy_prob),
        ('proxy_top1', proxy_rank == 1),
        (
            'proxy_top3',
            (proxy_rank > 0)
            & (proxy_rank <= 3),
        ),
        (
            'proxy_top5',
            (proxy_rank > 0)
            & (proxy_rank <= 5),
        ),

        # RRF summary
        ('rrf_global', rr[:, :4].sum(axis=1)),
        ('rrf_proxy', rr[:, 4:].sum(axis=1)),
        ('rrf_special', rr.sum(axis=1)),
        ('channel_count', (rr > 0).sum(axis=1)),
    ])

    data.update(
        (
            f'rr_{name}',
            rr[:, j],
        )
        for j, name in enumerate(SPECIAL_CHANNELS)
    )

    out = np.column_stack(
        list(data.values())
    ).astype('float32')
    out[~np.isfinite(out)] = -1.0

    return out, list(data)

retrieval.set_scope('full')
retrieval.set_proxy_lookup(
    proxy_lookups['fit'],
    proxy_tokens['fit'],
)

probe_regular = next(fit_q.itertuples(index=False))
probe_c, probe_r, probe_s = retrieval.retrieve_regular(probe_regular)
probe_x, REGULAR_FEATURE_NAMES = regular_features(
    probe_regular,
    probe_c,
    probe_r,
    probe_s,
)

F_REGULAR_RRF = REGULAR_FEATURE_NAMES.index('rrf_context')

if len(special_fit_q):
    probe_special = next(
        special_fit_q.itertuples(index=False)
    )
    sc, sr, ss = retrieval.retrieve_special(
        probe_special,
        global_depth=CFG['special_global_depth_grid'][0],
        proxy_topn=1,
        use_filter=True,
    )
    sx, SPECIAL_FEATURE_NAMES = special_features(
        probe_special,
        sc,
        sr,
        ss,
        proxy_lookups['fit'],
    )
else:
    raise RuntimeError('Нет special-fit контекстов')

F_SPECIAL_RRF = SPECIAL_FEATURE_NAMES.index('rrf_special')

atomic_json(
    V5_DIR / 'regular_feature_names.json',
    REGULAR_FEATURE_NAMES,
)
atomic_json(
    V5_DIR / 'special_feature_names.json',
    SPECIAL_FEATURE_NAMES,
)

In [14]:
def order_by(ids, scores):
    ids = np.asarray(ids, dtype='int32')
    return ids[
        np.lexsort(
            (ids, -np.asarray(scores))
        )
    ]

def rank_reciprocal(values, ids):
    values = np.asarray(values)
    ids = np.asarray(ids, dtype='int32')

    order = np.lexsort(
        (ids, -values)
    )

    out = np.empty(
        len(values),
        dtype='float32',
    )
    out[order] = (
        1
        / (
            CFG['rrf_k']
            + np.arange(1, len(values) + 1)
        )
    )
    return out

def ranking_metrics(ranking, positives):
    ranking = np.asarray(ranking, dtype='int32')
    assert len(ranking) == len(set(map(int, ranking)))

    rel = np.array(
        [
            i in positives
            for i in ranking[:50]
        ],
        dtype='float64',
    )

    hits = rel.sum()
    positions = np.flatnonzero(rel)
    n = len(positives)

    ideal = (
        1
        / np.log2(
            np.arange(min(n, 50)) + 2
        )
    ).sum()

    return {
        'Recall50': hits / n,
        'hits50': hits,
        'Hit50': float(hits > 0),
        'MRR50': (
            1 / (positions[0] + 1)
            if len(positions)
            else 0.0
        ),
        'nDCG50': float(
            (
                rel
                / np.log2(
                    np.arange(len(rel)) + 2
                )
            ).sum()
            / ideal
        ),
    }

def candidate_metrics(pool, positives):
    reached = len(
        set(map(int, pool))
        & positives
    )
    n = len(positives)

    return {
        'CandidateRecall': reached / n,
        'Oracle50': min(50, reached) / n,
    }



def summarize_eval(evaluation):
    table = (
        evaluation
        .groupby("method")
        .agg(
            Recall50=("Recall50", "mean"),
            HitRate50=("Hit50", "mean"),
            MRR50=("MRR50", "mean"),
            nDCG50=("nDCG50", "mean"),
            CandidateRecall=(
                "CandidateRecall",
                "mean",
            ),
            Oracle50=("Oracle50", "mean"),
            contexts=("context_id", "size"),
            hit_sum=("hits50", "sum"),
            positive_sum=("n_positive", "sum"),
        )
    )

    table["micro_Recall50"] = (
        table.hit_sum
        / table.positive_sum
    )

    return table.drop(
        columns=["hit_sum", "positive_sum"]
    )

### Результаты экспериментов с Candidate generation

| Конфигурация | Recall@50 | CandidateRecall | MRR@50 | nDCG@50 |
|---|---:|---:|---:|---:|
| BM25 global | 0.14571 | 0.39830 | 0.02860 | 0.05077 |
| E5 global | 0.15345 | 0.37954 | 0.03069 | 0.05403 |
| BM25 + E5 global | 0.16227 | 0.51222 | 0.03163 | 0.05634 |
| BM25 + context | 0.60280 | 0.79385 | 0.25889 | 0.32513 |
| E5 + context | 0.73699 | 0.83472 | 0.33513 | 0.41746 |
| BM25 + E5 + context | 0.73905 | 0.88993 | 0.32210 | 0.40668 |
| CatBoost reranking | 0.67212 | 0.88993 | 0.35590 | 0.41656 |
| Hybrid + CatBoost blend | **0.76616** | **0.88993** | **0.37619** | **0.45610** |

Главный прирост дал переход от глобального поиска к поиску с учётом контекста запроса.

При объединении BM25, E5 и контекста Candidate Recall вырос почти до `0.89`, то есть большая часть релевантных объявлений уже доходила до этапа Ranking.

Candidate Recall стал достаточно высоким, поэтому дальше улучшали итоговый Recall@50 за счет улучшения Ranking.

In [15]:
def load_cb(path):
    if not Path(path).exists():
        raise FileNotFoundError(path)
    model = CatBoostRanker()
    model.load_model(str(path))
    return model

regular_ranker = load_cb(
    V4_RUN_DIR
    / "regular_ranker_validation.cbm"
)
special_ranker = load_cb(
    V4_RUN_DIR
    / (
        "special_ranker_validation_"
        f"{SELECTED_SPECIAL_TOKEN}.cbm"
    )
)

regular_eval_ranker = load_cb(
    V4_RUN_DIR
    / "regular_ranker_holdout.cbm"
)
special_eval_ranker = load_cb(
    V4_RUN_DIR
    / (
        "special_ranker_holdout_"
        f"{SELECTED_SPECIAL_TOKEN}.cbm"
    )
)

final_regular_ranker = load_cb(
    V4_RUN_DIR
    / "regular_ranker_submission.cbm"
)
final_special_ranker = load_cb(
    V4_RUN_DIR
    / (
        "special_ranker_submission_"
        f"{SELECTED_SPECIAL_TOKEN}.cbm"
    )
)

In [16]:
def v4_rank_candidates(
    q,
    regular_model,
    special_model,
    proxy_lookup,
    topk,
):
    use_special = (
        USE_SPECIAL_EXPERT
        and not retrieval.has_exact_location(q)
    )

    if use_special:
        c, r, s = retrieval.retrieve_special(
            q,
            global_depth=BEST_SPECIAL_CONFIG[
                "global_depth"
            ],
            proxy_topn=BEST_SPECIAL_CONFIG[
                "proxy_topn"
            ],
            use_filter=BEST_SPECIAL_CONFIG[
                "use_filter"
            ],
        )

        x, _ = special_features(
            q,
            c,
            r,
            s,
            proxy_lookup,
        )

        pred = special_model.predict(
            x,
            thread_count=CFG[
                "ranker_threads"
            ],
        )

        base_score = (
            BEST_SPECIAL_ALPHA
            * rank_reciprocal(
                pred,
                c,
            )
            + (1 - BEST_SPECIAL_ALPHA)
            * rank_reciprocal(
                x[:, F_SPECIAL_RRF],
                c,
            )
        )

    else:
        c, r, s = retrieval.retrieve_regular(q)

        x, _ = regular_features(
            q,
            c,
            r,
            s,
        )

        pred = regular_model.predict(
            x,
            thread_count=CFG[
                "ranker_threads"
            ],
        )

        base_score = (
            CFG["regular_alpha"]
            * rank_reciprocal(
                pred,
                c,
            )
            + (1 - CFG["regular_alpha"])
            * rank_reciprocal(
                x[:, F_REGULAR_RRF],
                c,
            )
        )

    c = np.asarray(
        c,
        dtype="int32",
    )
    base_score = np.asarray(
        base_score,
        dtype="float32",
    )

    order = np.lexsort(
        (c, -base_score)
    )
    order = order[
        :min(int(topk), len(order))
    ]

    return (
        c[order],
        base_score[order],
        bool(use_special),
    )

## 6. Cross-encoder reranking

Cross-Encoder решает другую задачу: он получает пару `query + item` одновременно и оценивает релевантность конкретного объявления конкретному запросу.

Из-за высокой вычислительной стоимости Cross-Encoder применяется не ко всем объявлениям, а только к кандидатам, найденным на предыдущем этапе.

Чтобы отдельно измерить влияние Ranking, все варианты сравнивались на одном и том же наборе кандидатов.

| Модель | Recall@50 | CandidateRecall | Прирост к базовому ranking |
|---|---:|---:|---:|
| Base hybrid ranker | 0.77598 | 0.84041 | — |
| Zero-shot BGE | 0.79669 | 0.84041 | +0.02072 |
| Fine-tuned BGE | **0.81186** | 0.84041 | **+0.03588** |

Candidate Recall во всех строках одинаковый, поэтому прирост Recall@50 связан именно с улучшением Ranking.

Сначала использовалась готовая BGE reranker модель, затем она была дообучена на наших данных.

## 7. Выбор размера candidate pool

После улучшения Ranking исследовалось число кандидатов, передаваемых в Cross-Encoder.

При увеличении `K` растёт Candidate Recall: релевантное объявление с большей вероятностью попадает в candidate pool.

Но одновременно задача Ranking становится сложнее, так как Cross-Encoder должен выбрать Top-50 среди большего числа похожих объявлений.

| K | Recall@50 | CandidateRecall |
|---:|---:|---:|
| 100 | 0.811859 | 0.840406 |
| 150 | 0.819716 | 0.869276 |
| 200 | **0.820250** | 0.881356 |
| 300 | 0.817716 | **0.896223** |

Отдельно были рассмотрены две группы запросов:

| Сегмент | K=100 | K=150 | K=200 | K=300 |
|---|---:|---:|---:|---:|
| Regular Recall@50 | 0.837114 | **0.845267** | 0.844034 | 0.839028 |
| Special Recall@50 | 0.649206 | 0.655159 | 0.667063 | **0.680456** |
| Regular CandidateRecall | 0.863762 | 0.889798 | 0.899595 | 0.908760 |
| Special CandidateRecall | 0.689980 | 0.737103 | 0.763889 | **0.815476** |

Для regular запросов увеличение `K` выше `150` почти не помогает итоговой метрике.

Для special запросов, наоборот, основной проблемой остаётся поиск кандидатов, поэтому им выгодно передавать более широкий candidate pool.



## 8. Проверенные усложнения

После этого были проверены дополнительные варианты:

- другие веса объединения ranking-сигналов;
- динамический выбор `K`;
- использование исторических совпадений;
- дополнительная meta-model для объединения результатов.

На локальной проверке некоторые варианты давали небольшой прирост, но он не подтверждался достаточно стабильно.

Поэтому в финальном решении оставлена более простая схема.

## 9. Финальная архитектура

После проведённых экспериментов была выбрана следующая схема:

**Query → BM25 + E5 + context-aware Candidate generation → CatBoost Ranking → BGE Cross-Encoder → объединение двух Ranking → Top-50**

Дальше в notebook эта конфигурация обучается и проверяется без дополнительного подбора гиперпараметров.

## 10. Обучающая выборка Cross-Encoder

Для финального запуска выбираются:

- до `40 000` regular contexts;
- до `25 000` special contexts;
- до двух positive объявлений на один context;
- три hard negative на каждый positive.

In [17]:
fit_all_q = get_contexts(fit_pairs)
regular_available = fit_all_q.loc[fit_all_q.location.isin(FULL_LOCATION_SET)].reset_index(drop=True)
special_available = special_contexts(fit_all_q, FULL_LOCATION_SET)

regular_train_q = choose_contexts(
    regular_available, CFG["train_regular_contexts"], CFG["seed"] + 501
)
special_train_q = choose_contexts(
    special_available, CFG["train_special_contexts"], CFG["seed"] + 502
)
v5_train_q = pd.concat([regular_train_q, special_train_q], ignore_index=True)
v5_valid_q = tune_q.reset_index(drop=True)
holdout_5000 = choose_contexts(
    holdout_q, min(CFG["holdout_contexts"], len(holdout_q)), CFG["fresh_holdout_seed"] + 451
)

assert v5_train_q.context_id.is_unique
assert not set(v5_train_q.query_norm) & set(v5_valid_q.query_norm)
assert not set(v5_train_q.query_norm) & set(holdout_5000.query_norm)
assert len(regular_train_q) > 0 and len(special_train_q) > 0

training_selection = pd.DataFrame({
    "segment": ["regular", "special", "total"],
    "contexts": [len(regular_train_q), len(special_train_q), len(v5_train_q)],
})
display(training_selection)

,segment,contexts
0,regular,40000
1,special,25000
2,total,65000


In [18]:
train_unique_texts = (
    v5_train_q["query_text"]
    .astype(str)
    .drop_duplicates()
    .tolist()
)

missing_query_texts = [
    text
    for text in train_unique_texts
    if text not in query_text_to_row
]


if missing_query_texts:
    e5_query_encoder = SentenceTransformer(
        "intfloat/multilingual-e5-base",
        device="cuda",
    )
    e5_query_encoder.max_seq_length = CFG["max_seq_length"]

    # Detect the cached query-input convention on known queries.
    probe_texts = list(query_text_to_row.keys())[: min(32, len(query_text_to_row))]
    probe_rows = np.array(
        [query_text_to_row[t] for t in probe_texts],
        dtype=np.int64,
    )
    cached_probe = np.asarray(
        query_vectors[probe_rows],
        dtype=np.float32,
    )
    cached_probe /= np.clip(
        np.linalg.norm(cached_probe, axis=1, keepdims=True),
        1e-12,
        None,
    )

    probe_prefixed = e5_query_encoder.encode(
        ["query: " + t for t in probe_texts],
        batch_size=64,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    probe_raw = e5_query_encoder.encode(
        probe_texts,
        batch_size=64,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    cos_prefixed = np.sum(cached_probe * probe_prefixed, axis=1)
    cos_raw = np.sum(cached_probe * probe_raw, axis=1)

    mean_prefixed = float(cos_prefixed.mean())
    mean_raw = float(cos_raw.mean())


    if mean_prefixed >= mean_raw:
        encode_prefix = "query: "
        best_probe_cos = mean_prefixed
    else:
        encode_prefix = ""
        best_probe_cos = mean_raw

    # A large mismatch means the cached embeddings used a different encoder recipe.
    if best_probe_cos < 0.99:
        raise RuntimeError(
            "Не удалось безопасно воспроизвести cached E5 query embeddings. "
            f"Best probe cosine={best_probe_cos:.6f}"
        )

    extra_query_vectors_path = (
        V5_DIR / "extra_train_query_vectors.npy"
    )
    extra_query_texts_path = (
        V5_DIR / "extra_train_query_texts.json"
    )

    if (
        extra_query_vectors_path.exists()
        and extra_query_texts_path.exists()
    ):
        saved_texts = json.loads(
            extra_query_texts_path.read_text(
                encoding="utf-8"
            )
        )
        if saved_texts != missing_query_texts:
            raise RuntimeError(
                "Saved extra E5 query cache belongs to another text set."
            )
        extra_vectors = np.load(
            extra_query_vectors_path
        ).astype("float32")
    else:
        texts_for_encoder = [
            encode_prefix + text
            for text in missing_query_texts
        ]
        extra_vectors = e5_query_encoder.encode(
            texts_for_encoder,
            batch_size=128,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
        ).astype("float32")

        np.save(
            extra_query_vectors_path,
            extra_vectors,
        )
        extra_query_texts_path.write_text(
            json.dumps(
                missing_query_texts,
                ensure_ascii=False,
            ),
            encoding="utf-8",
        )

    base_query_vectors = np.asarray(
        query_vectors,
        dtype=np.float32,
    )

    old_n = len(base_query_vectors)

    query_vectors = np.concatenate(
        [
            base_query_vectors,
            extra_vectors,
        ],
        axis=0,
    )

    for offset, text in enumerate(
        missing_query_texts
    ):
        query_text_to_row[text] = old_n + offset

    assert all(
        text in query_text_to_row
        for text in train_unique_texts
    )


    del e5_query_encoder
    del probe_prefixed
    del probe_raw
    del cached_probe
    del extra_vectors
    del base_query_vectors
    gc.collect()
    torch.cuda.empty_cache()

else:
    pass

Batches: 100%|██████████| 151/151 [00:02<00:00, 57.69it/s]


In [19]:
assert all(text in query_text_to_row for text in v5_train_q.query_text)
assert len(query_vectors) == len(query_text_to_row)
assert query_vectors.shape[1] == document_vectors.shape[1]
assert np.isfinite(np.asarray(query_vectors[[query_text_to_row[t] for t in v5_train_q.query_text[:16]]])).all()


## 11. Подготовка кандидатов для обучения и проверки

In [20]:
import shutil

def digest_obj(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False, default=str).encode()).hexdigest()

def frame_digest(df, columns):
    x = df[columns].reset_index(drop=True).copy()
    return hashlib.sha256(pd.util.hash_pandas_object(x, index=True).values.tobytes()).hexdigest()

def persist_manifest(directory, identity):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    p = directory / "manifest.json"
    if p.exists():
        if json.loads(p.read_text()) != identity:
            raise RuntimeError(f"Stale cache identity: {p}")
    else:
        atomic_json(p, identity)
    return directory

def atomic_parquet(df, path):
    path = Path(path)
    tmp = path.with_suffix(".tmp.parquet")
    df.to_parquet(tmp, index=False)
    os.replace(tmp, path)

def dataset_positive_digest(frame):
    keys = set(frame.context_id.astype(str))
    pairs_in = pairs.loc[pairs.context_id.astype(str).isin(keys), ["context_id", "item_row", "pair_count"]]
    return frame_digest(pairs_in.sort_values(["context_id", "item_row"]), ["context_id", "item_row", "pair_count"])

CFG_ID = digest_obj(V5_SIGNATURE)
TRAIN_FRAME_ID = frame_digest(v5_train_q, ["context_id", "query_norm", "query_text", "location", "category", "params", "delivery"])
TRAIN_LABEL_ID = dataset_positive_digest(v5_train_q)
MODEL_FILES = {
    "validation_regular": file_sha256(V4_RUN_DIR / "regular_ranker_validation.cbm"),
    "validation_special": file_sha256(V4_RUN_DIR / f"special_ranker_validation_{SELECTED_SPECIAL_TOKEN}.cbm"),
    "holdout_regular": file_sha256(V4_RUN_DIR / "regular_ranker_holdout.cbm"),
    "holdout_special": file_sha256(V4_RUN_DIR / f"special_ranker_holdout_{SELECTED_SPECIAL_TOKEN}.cbm"),
    "benchmark_regular": file_sha256(V4_RUN_DIR / "regular_ranker_submission.cbm"),
    "benchmark_special": file_sha256(V4_RUN_DIR / f"special_ranker_submission_{SELECTED_SPECIAL_TOKEN}.cbm"),
}

In [21]:
CANDIDATE_VERSION = "final_adaptive_candidates_v1"
QUERY_COLUMNS = ["query_norm", "query_text", "params", "location", "category", "delivery"]

def candidate_folder(frame, tag, scope, model_regular, model_special, proxy_key):
    key_col = "query_id" if "query_id" in frame else "context_id"
    tag_id = {
        "version": CANDIDATE_VERSION, "cfg_id": CFG_ID,
        "frame": frame_digest(frame, [key_col] + QUERY_COLUMNS),
        "rankers": [MODEL_FILES[f"{tag}_regular"], MODEL_FILES[f"{tag}_special"]],
        "proxy": proxy_tokens[proxy_key], "scope": scope,
        "depth": [CFG["regular_K"], CFG["special_K"]],
    }
    return persist_manifest(V5_DIR / f"candidates_{tag}", tag_id)

def build_adaptive_candidates(frame, tag, scope, model_regular, model_special, proxy_key, chunk=128):
    folder = candidate_folder(frame, tag, scope, model_regular, model_special, proxy_key)
    key_col = "query_id" if "query_id" in frame else "context_id"
    ordered = frame.reset_index(drop=True)
    n_parts = math.ceil(len(ordered) / chunk)
    files = [folder / f"part_{i:05d}.parquet" for i in range(n_parts)]
    if all(p.exists() for p in files):
        return folder
    load_gpu_documents()
    retrieval.cache.clear()
    retrieval.set_scope(scope)
    retrieval.set_proxy_lookup(proxy_lookups[proxy_key], proxy_tokens[proxy_key])
    try:
        for idx in tqdm(range(n_parts), desc=f"candidates {tag}"):
            part = ordered.iloc[idx*chunk:(idx+1)*chunk]
            expected_keys = part[key_col].astype(str).tolist()
            if files[idx].exists():
                saved = pd.read_parquet(files[idx], columns=["key", "item_row", "base_rank"])
                assert set(saved.key.astype(str)) == set(expected_keys)
                assert saved.groupby("key").base_rank.min().eq(1).all()
                assert not saved.duplicated(["key", "item_row"]).any()
                continue
            rows = []
            for q in part.itertuples(index=False):
                ids, base, special = v4_rank_candidates(
                    q, model_regular, model_special, proxy_lookups[proxy_key], CFG["special_K"]
                )
                limit = CFG["special_K"] if special else CFG["regular_K"]
                ids, base = ids[:limit], base[:limit]
                assert len(ids) >= 50 and len(np.unique(ids)) == len(ids)
                assert ids.min() >= 0 and ids.max() < (BENCH_N if scope == "benchmark" else len(items))
                key = str(getattr(q, key_col))
                rows.extend((key, int(d), float(s), rank, bool(special))
                            for rank, (d, s) in enumerate(zip(ids, base), 1))
            artifact = pd.DataFrame(rows, columns=["key", "item_row", "base_score", "base_rank", "special_used"])
            assert set(artifact.key) == set(expected_keys)
            atomic_parquet(artifact, files[idx])
    finally:
        retrieval.cache.clear()
    return folder

def read_candidate_parts(folder):
    files = sorted(Path(folder).glob("part_*.parquet"))
    if not files:
        raise RuntimeError(f"Нет кандидатов: {folder}")
    out = pd.concat((pd.read_parquet(p) for p in files), ignore_index=True)
    out["key"] = out.key.astype(str)
    assert not out.duplicated(["key", "item_row"]).any()
    assert out.groupby("key").base_rank.min().eq(1).all()
    assert out.groupby("key").size().ge(50).all()
    assert out.groupby("key").special_used.nunique().eq(1).all()
    return out

In [22]:
INNER_CAND_DIR = build_adaptive_candidates(
    v5_valid_q, "validation", "full", regular_ranker, special_ranker, "fit"
)
inner_cand = read_candidate_parts(INNER_CAND_DIR)
assert inner_cand.key.nunique() == len(v5_valid_q)

coverage = []
for key, group in inner_cand.groupby("key", sort=False):
    p = positive_rows[key]
    coverage.append({
        "context_id": key,
        "special_used": bool(group.special_used.iloc[0]),
        **candidate_metrics(group.item_row.to_numpy("int32"), p),
    })
inner_coverage = pd.DataFrame(coverage)

candidate_summary = pd.concat([
    inner_coverage.assign(segment="overall").groupby("segment")[["CandidateRecall", "Oracle50"]].mean(),
    inner_coverage.assign(segment=np.where(inner_coverage.special_used, "special", "regular"))
        .groupby("segment")[["CandidateRecall", "Oracle50"]].mean(),
]).round(6)
display(candidate_summary)

atomic_parquet(inner_coverage, V5_DIR / "inner_candidate_coverage.parquet")
del inner_cand, coverage
gc.collect()

candidates validation: 100%|██████████| 20/20 [02:20<00:00,  7.02s/it]


,CandidateRecall,Oracle50
segment,,
overall,0.879810,0.879810
regular,0.889798,0.889798
special,0.815476,0.815476


158

## 12. Hard-negative mining

Для каждого context:

1. строится Top-100 базового ranking;
2. выбирается до двух известных positive объявлений;
3. для каждого positive выбираются три negative из разных частей ranking.

Таким образом Cross-Encoder учится различать действительно похожие объявления.

In [24]:
FIT_LABEL_ID = frame_digest(
    fit_pairs[["context_id", "item_row"]].sort_values(["context_id", "item_row"]),
    ["context_id", "item_row"],
)
mining_identity = {
    "version": "two_pos_three_neg_top100_v1",
    "cfg": CFG_ID,
    "frame": TRAIN_FRAME_ID,
    "labels": FIT_LABEL_ID,
    "rankers": [MODEL_FILES["validation_regular"], MODEL_FILES["validation_special"]],
    "proxy_fit": proxy_tokens["fit"],
    "regular_depth": CFG["mine_regular_depth"],
    "special_depth": CFG["mine_special_depth"],
    "max_pos_per_context": CFG["max_pos_per_context"],
    "negatives_per_positive": CFG["negatives_per_positive"],
}
MINE_DIR = persist_manifest(V5_DIR / "mined_pairs", mining_identity)

selected_ids = set(v5_train_q.context_id.astype(str))
pair_priority = (
    pairs.loc[pairs.context_id.astype(str).isin(selected_ids),
              ["context_id", "item_row", "pair_count"]]
    .sort_values(["context_id", "pair_count", "item_row"], ascending=[True, False, True])
    .groupby("context_id", sort=False)
    .item_row.agg(list).to_dict()
)
assert len(pair_priority) == len(v5_train_q)


def select_negative_rows(ranked_candidates, positives, context_id, n):
    eligible = [(rank, int(item)) for rank, item in enumerate(ranked_candidates, 1)
                if int(item) not in positives]
    if not eligible:
        raise RuntimeError(f"Нет negatives для {context_id}")

    seed = int.from_bytes(
        hashlib.blake2b(str(context_id).encode(), digest_size=8).digest(), "little"
    ) ^ CFG["seed"]
    rng = np.random.default_rng(seed)

    selected = [eligible[0]]
    # medium and deep negatives, matching the selected top-100 training recipe
    for lo, hi in [(5, min(25, len(eligible))), (25, min(100, len(eligible)))]:
        pool = [x for x in eligible[lo:hi] if x[1] not in {item for _, item in selected}]
        if pool and len(selected) < n:
            selected.append(pool[int(rng.integers(len(pool)))])

    for rank, item in eligible:
        if len(selected) >= n:
            break
        if item not in {x for _, x in selected}:
            selected.append((rank, item))
    return selected[:n]


ordered_train = v5_train_q.sort_values(["query_text", "context_id"]).reset_index(drop=True)
chunk = 128
part_files = [
    MINE_DIR / f"part_{i:05d}.parquet"
    for i in range(math.ceil(len(ordered_train) / chunk))
]

if not all(p.exists() for p in part_files):
    load_gpu_documents()
    retrieval.cache.clear()
    retrieval.set_scope("full")
    retrieval.set_proxy_lookup(proxy_lookups["fit"], proxy_tokens["fit"])

    for i in tqdm(range(len(part_files)), desc="hard-negative mining"):
        part = ordered_train.iloc[i * chunk:(i + 1) * chunk]
        if part_files[i].exists():
            old = pd.read_parquet(part_files[i], columns=["context_id", "pos_row", "neg_row"])
            assert set(old.context_id.astype(str)) == set(part.context_id.astype(str))
            counts = old.groupby("context_id").size()
            assert counts.isin([3, 6]).all()
            continue

        records = []
        for q in part.itertuples(index=False):
            depth = CFG["mine_special_depth"] if not retrieval.has_exact_location(q) else CFG["mine_regular_depth"]
            ranked, _, special_used = v4_rank_candidates(
                q, regular_ranker, special_ranker, proxy_lookups["fit"], depth
            )
            positives = positive_rows[q.context_id]
            pos_order = [
                int(x) for x in pair_priority[q.context_id] if int(x) in positives
            ][:CFG["max_pos_per_context"]]
            negatives = select_negative_rows(
                ranked, positives, q.context_id, CFG["negatives_per_positive"]
            )

            for pos_row in pos_order:
                for neg_rank, neg_row in negatives:
                    records.append((
                        str(q.context_id), str(q.query_text), int(pos_row), int(neg_row),
                        int(neg_rank), bool(special_used)
                    ))

        artifact = pd.DataFrame(
            records,
            columns=["context_id", "query_text", "pos_row", "neg_row", "neg_rank", "special_used"],
        ).drop_duplicates(["context_id", "pos_row", "neg_row"])
        atomic_parquet(artifact, part_files[i])
    retrieval.cache.clear()

free_gpu_documents()
train_triplets = pd.concat((pd.read_parquet(p) for p in part_files), ignore_index=True)
assert train_triplets.context_id.nunique() == len(v5_train_q)
assert not train_triplets.duplicated(["context_id", "pos_row", "neg_row"]).any()
triplets_per_context = train_triplets.groupby("context_id").size()
assert triplets_per_context.isin([3, 6]).all()

training_pairs_summary = pd.DataFrame({
    "metric": ["contexts", "pairwise examples", "contexts with 1 positive", "contexts with 2 positives"],
    "value": [
        train_triplets.context_id.nunique(),
        len(train_triplets),
        int((triplets_per_context == 3).sum()),
        int((triplets_per_context == 6).sum()),
    ],
})
display(training_pairs_summary)

del pair_priority, ordered_train, triplets_per_context
gc.collect()

hard-negative mining: 100%|██████████| 508/508 [29:46<00:00,  3.52s/it] 


,metric,value
0,contexts,65000
1,pairwise examples,207351
2,contexts with 1 positive,60883
3,contexts with 2 positives,4117


0

## 13. Fine-tuning BGE reranker

Один обучающий пример имеет вид:

**query + positive item + negative item**

Модель дает positive объявлению более высокий score, чем hard negative.

Для уменьшения числа параметров используется **LoRA**.

In [25]:
CE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    CFG["ce_model"],
)

base_ce_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        CFG["ce_model"],
        torch_dtype=CE_DTYPE,
    )
    .to("cuda")
)

base_ce_model.config.use_cache = False

module_leaf_names = {
    name.split(".")[-1]
    for name, _ in base_ce_model.named_modules()
}

if {
    "query",
    "key",
    "value",
}.issubset(module_leaf_names):
    target_modules = [
        "query",
        "key",
        "value",
    ]
elif {
    "q_proj",
    "k_proj",
    "v_proj",
}.issubset(module_leaf_names):
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
    ]
else:
    raise RuntimeError(
        "Не удалось автоматически найти "
        "attention projection modules."
    )

head_name = None
for candidate in [
    "classifier",
    "score",
]:
    if hasattr(base_ce_model, candidate):
        head_name = candidate
        break


assert head_name is not None
base_ce_model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1940.32it/s]


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=1024, out_features=1024, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=1024, out_features=1, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(8194, 1024, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
  

In [26]:
def base_model_digest(model):
    h = hashlib.sha256()
    for name, weights in model.state_dict().items():
        h.update(name.encode())
        h.update(str((tuple(weights.shape), str(weights.dtype))).encode())
        raw = weights.detach().contiguous().cpu().reshape(-1).view(torch.uint8).numpy()
        h.update(raw.tobytes())
        del raw
    return h.hexdigest()

BASE_MODEL_HASH = base_model_digest(base_ce_model)
TOKENIZER_HASH = hashlib.sha256(tokenizer.backend_tokenizer.to_str().encode()).hexdigest()


In [27]:
import shutil

required_before_training = [
    *RAW_PATHS.values(),
    V4_RUN_DIR / "documents.npy",
    V4_RUN_DIR / "queries.npy",
]
assert all(p.is_file() for p in required_before_training)
assert not set(v5_train_q.query_norm) & set(v5_valid_q.query_norm)
assert not set(v5_train_q.query_norm) & set(holdout_5000.query_norm)
assert train_triplets.context_id.nunique() == len(v5_train_q)
assert train_triplets.groupby("context_id").size().isin([3, 6]).all()
assert inner_coverage.CandidateRecall.between(0, 1).all()
assert torch.cuda.is_available()

free_gb = shutil.disk_usage(V5_DIR).free / 2**30
assert free_gb >= CFG["min_free_disk_gb"], f"Only {free_gb:.1f} GiB free"
probe_path = V5_DIR / ".write_probe"
probe_path.write_text("ok")
probe_path.unlink()

In [28]:
class PairwiseDataset(Dataset):
    def __init__(self, triplets):
        self.rows = triplets.reset_index(drop=True)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows.iloc[int(idx)]
        return (str(row.query_text), str(items.text.iat[int(row.pos_row)]),
                str(items.text.iat[int(row.neg_row)]))

def pairwise_collate(batch):
    query = [x[0] for x in batch]
    positive = [x[1] for x in batch]
    negative = [x[2] for x in batch]
    # Positive logits are first; negative logits start at len(batch).
    enc = tokenizer(query + query, positive + negative, padding=True,
                    truncation=True, max_length=CFG["ce_max_length"], return_tensors="pt")
    return enc, len(batch)

In [29]:
TRIPLET_ID = frame_digest(
    train_triplets, ["context_id", "query_text", "pos_row", "neg_row", "neg_rank", "special_used"]
)
TRAIN_IDENTITY = {
    "run": CFG_ID, "triplets": TRIPLET_ID,
    "model": CFG["ce_model"], "base_commit": getattr(base_ce_model.config, "_commit_hash", None),
    "base_sha256": BASE_MODEL_HASH, "tokenizer_sha256": TOKENIZER_HASH,
    "max_length": CFG["ce_max_length"], "batch": CFG["ce_train_batch_pairs"],
    "accum": CFG["ce_grad_accum"], "seed": CFG["seed"],
}
TRAIN_ID = digest_obj(TRAIN_IDENTITY)
ADAPTER_DIR = V5_DIR / "ce_adapter_final"
TRAINING_DONE = V5_DIR / "training_completed.json"
CKPT_DIR = persist_manifest(V5_DIR / "training_checkpoints", TRAIN_IDENTITY)
train_dataset = PairwiseDataset(train_triplets)
train_total = len(train_dataset)
optimizer_steps = math.ceil(train_total / (CFG["ce_train_batch_pairs"] * CFG["ce_grad_accum"]))
warmup_steps = int(optimizer_steps * CFG["ce_warmup_ratio"])

if TRAINING_DONE.exists():
    if json.loads(TRAINING_DONE.read_text()).get("identity") != TRAIN_ID or not ADAPTER_DIR.exists():
        raise RuntimeError("Сохранённый adapter не соответствует текущему run.")
    ce_model = PeftModel.from_pretrained(base_ce_model, ADAPTER_DIR, is_trainable=False).to("cuda").eval()
    training_history = pd.read_csv(V5_DIR / "training_log.csv").to_dict("records")
else:
    snapshots = sorted(
        [p for p in CKPT_DIR.glob("step_*")
         if p.is_dir() and re.fullmatch(r"step_\d+", p.name) and (p / "commit.json").exists()],
        key=lambda p: int(p.name.split("_")[-1])
    )
    if snapshots:
        checkpoint = snapshots[-1]
        commit = json.loads((checkpoint / "commit.json").read_text())
        if commit["identity"] != TRAIN_ID:
            raise RuntimeError("Checkpoint принадлежит другому run.")
        ce_model = PeftModel.from_pretrained(base_ce_model, checkpoint, is_trainable=True).to("cuda")
    else:
        leafs = {name.split(".")[-1] for name, _ in base_ce_model.named_modules()}
        if {"query", "key", "value"} <= leafs:
            target_modules = ["query", "key", "value"]
        elif {"q_proj", "k_proj", "v_proj"} <= leafs:
            target_modules = ["q_proj", "k_proj", "v_proj"]
        else:
            raise RuntimeError("Не обнаружены attention projections модели BGE.")
        head_name = next((n for n in ("classifier", "score") if hasattr(base_ce_model, n)), None)
        if head_name is None:
            raise RuntimeError("Не обнаружена trainable classification head модели BGE.")
        lora_cfg = LoraConfig(
            task_type=TaskType.SEQ_CLS, r=CFG["ce_lora_r"],
            lora_alpha=CFG["ce_lora_alpha"], lora_dropout=CFG["ce_lora_dropout"],
            target_modules=target_modules, modules_to_save=[head_name], bias="none"
        )
        ce_model = get_peft_model(base_ce_model, lora_cfg)
    if hasattr(ce_model, "enable_input_require_grads"):
        ce_model.enable_input_require_grads()
    ce_model.gradient_checkpointing_enable()
    trainable = [p for p in ce_model.parameters() if p.requires_grad]
    assert 0 < sum(p.numel() for p in trainable) < sum(p.numel() for p in ce_model.parameters())
    if not snapshots:
        # Real mini-batch checks tokenizer, LoRA forward/backward and finite gradients.
        preflight_batch = [train_dataset[int(i)] for i in range(min(4, train_total))]
        preflight_enc, preflight_pairs = pairwise_collate(preflight_batch)
        preflight_enc = {k: v.to("cuda") for k, v in preflight_enc.items()}
        ce_model.train()
        with torch.autocast(device_type="cuda", dtype=CE_DTYPE):
            preflight_logits = ce_model(**preflight_enc).logits.reshape(-1)
            assert len(preflight_logits) == 2 * preflight_pairs
            preflight_loss = F.softplus(-(
                preflight_logits[:preflight_pairs] - preflight_logits[preflight_pairs:]
            )).mean()
        assert bool(torch.isfinite(preflight_loss))
        preflight_loss.backward()
        lo_grads = [p.grad for n, p in ce_model.named_parameters()
                    if p.requires_grad and "lora_" in n and p.grad is not None]
        assert lo_grads and all(bool(torch.isfinite(g).all()) for g in lo_grads)
        ce_model.zero_grad(set_to_none=True)
        del preflight_batch, preflight_enc, preflight_logits, preflight_loss, lo_grads
        torch.cuda.empty_cache()
        random.seed(CFG["seed"])
        np.random.seed(CFG["seed"])
        torch.manual_seed(CFG["seed"])
        torch.cuda.manual_seed_all(CFG["seed"])
    optimizer = torch.optim.AdamW(trainable, lr=CFG["ce_learning_rate"],
                                  weight_decay=CFG["ce_weight_decay"])
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=optimizer_steps
    )
    samples_seen, opt_step, training_history = 0, 0, []
    if snapshots:
        state = torch.load(checkpoint / "state.pt", map_location="cpu", weights_only=False)
        if state["identity"] != TRAIN_ID:
            raise RuntimeError("Checkpoint state отличается от модели.")
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        samples_seen, opt_step = int(state["samples_seen"]), int(state["opt_step"])
        training_history = state["history"]
        assert 0 <= samples_seen <= train_total and 0 <= opt_step <= optimizer_steps
        random.setstate(state["python_rng"])
        np.random.set_state(state["numpy_rng"])
        torch.set_rng_state(state["torch_rng"])
        torch.cuda.set_rng_state_all(state["cuda_rng"])

    def save_checkpoint():
        dest = CKPT_DIR / f"step_{opt_step:07d}"
        if dest.exists():
            return
        tmp = CKPT_DIR / f"step_{opt_step:07d}.tmp"
        if tmp.exists():
            shutil.rmtree(tmp)
        tmp.mkdir()
        ce_model.save_pretrained(tmp)
        torch.save({
            "identity": TRAIN_ID, "samples_seen": samples_seen, "opt_step": opt_step,
            "history": training_history, "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(), "python_rng": random.getstate(),
            "numpy_rng": np.random.get_state(), "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all(),
        }, tmp / "state.pt")
        atomic_json(tmp / "commit.json", {"identity": TRAIN_ID, "opt_step": opt_step, "samples_seen": samples_seen})
        os.replace(tmp, dest)
        finished = sorted([p for p in CKPT_DIR.glob("step_*")
                           if p.is_dir() and re.fullmatch(r"step_\d+", p.name) and (p / "commit.json").exists()])
        for previous in finished[:-2]:
            shutil.rmtree(previous)

    # Fixed permutation; a saved sample cursor resumes at the next microbatch.
    permutation = np.random.default_rng(CFG["seed"] + 815).permutation(len(train_dataset))[:train_total]
    remaining = torch.utils.data.Subset(train_dataset, permutation[samples_seen:].tolist())
    loader = DataLoader(remaining, batch_size=CFG["ce_train_batch_pairs"], shuffle=False,
                        num_workers=0, collate_fn=pairwise_collate, pin_memory=True,
                        generator=torch.Generator().manual_seed(CFG["seed"] + 816))
    ce_model.train()
    optimizer.zero_grad(set_to_none=True)
    accum = CFG["ce_grad_accum"]
    since_step, last_losses = 0, []
    total_batches_remaining = len(loader)
    for batch_idx, (enc, pair_count) in tqdm(enumerate(loader), total=total_batches_remaining, desc="clean CE train"):
        enc = {k: v.to("cuda", non_blocking=True) for k, v in enc.items()}
        tail_batches = total_batches_remaining - (batch_idx // accum) * accum
        divisor = min(accum, tail_batches)
        with torch.autocast(device_type="cuda", dtype=CE_DTYPE):
            logits = ce_model(**enc).logits.reshape(-1)
            assert logits.numel() == pair_count * 2
            raw_loss = F.softplus(-(logits[:pair_count] - logits[pair_count:])).mean()
            loss = raw_loss / divisor
        if not bool(torch.isfinite(raw_loss).item()):
            raise FloatingPointError(f"Non-finite RankNet loss at optimizer step {opt_step}.")
        loss.backward()
        last_losses.append(float(raw_loss.detach().float().cpu()))
        samples_seen += pair_count
        since_step += 1
        if since_step == accum or batch_idx + 1 == total_batches_remaining:
            grad_norm = float(torch.nn.utils.clip_grad_norm_(trainable, CFG["ce_max_grad_norm"]).detach().float().cpu())
            if not np.isfinite(grad_norm):
                raise FloatingPointError(f"Non-finite gradient at optimizer step {opt_step}.")
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            opt_step += 1
            since_step = 0
            training_history.append({"step": opt_step, "samples_seen": samples_seen,
                                     "loss": float(np.mean(last_losses)), "grad_norm": grad_norm,
                                     "lr": float(scheduler.get_last_lr()[0])})
            last_losses = []
            if opt_step % CFG["checkpoint_optimizer_steps"] == 0:
                save_checkpoint()
    assert samples_seen == train_total and opt_step == optimizer_steps
    save_checkpoint()
    ce_model.eval()
    if ADAPTER_DIR.exists():
        # An incomplete adapter was left before the completion marker; the checkpoint above is authoritative.
        shutil.rmtree(ADAPTER_DIR)
    tmp_adapter = V5_DIR / "ce_adapter_final.tmp"
    if tmp_adapter.exists():
        shutil.rmtree(tmp_adapter)
    tmp_adapter.mkdir()
    ce_model.save_pretrained(tmp_adapter)
    tokenizer.save_pretrained(tmp_adapter)
    os.replace(tmp_adapter, ADAPTER_DIR)
    atomic_parquet(pd.DataFrame(training_history), V5_DIR / "training_log.parquet")
    pd.DataFrame(training_history).to_csv(V5_DIR / "training_log.csv", index=False)
    atomic_json(TRAINING_DONE, {
        "identity": TRAIN_ID, "triplets": train_total, "contexts": len(v5_train_q),
        "regular_contexts": len(regular_train_q), "special_contexts": len(special_train_q),
        "optimizer_steps": opt_step, "adapter": str(ADAPTER_DIR),
    })

# Reload the saved adapter from disk before evaluation.
if "optimizer" in globals():
    del optimizer, scheduler, trainable
    gc.collect()
del ce_model, base_ce_model
gc.collect()
torch.cuda.empty_cache()
reload_base = AutoModelForSequenceClassification.from_pretrained(
    CFG["ce_model"], torch_dtype=CE_DTYPE
).to("cuda")
ce_model = PeftModel.from_pretrained(reload_base, ADAPTER_DIR, is_trainable=False).to("cuda").eval()
assert len(training_history) > 0
training_curve_summary = pd.DataFrame({
    "window": ["first 10 optimizer steps", "last 10 optimizer steps"],
    "mean_pairwise_loss": [
        np.mean([x["loss"] for x in training_history[:10]]),
        np.mean([x["loss"] for x in training_history[-10:]]),
    ],
})
display(training_curve_summary.round(6))
if "optimizer" in globals():
    del optimizer, scheduler, trainable
if "train_triplets" in globals():
    del train_triplets, train_dataset
gc.collect()
torch.cuda.empty_cache()

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 2021.38it/s]


,window,mean_pairwise_loss
0,first 10 optimizer steps,0.844486
1,last 10 optimizer steps,0.470659


## 14. Validation

После обучения модель проверяется на Validation.

Для каждого запроса:

1. строится candidate pool;
2. для regular запросов берётся Top-150, для special — Top-300;
3. Cross-Encoder рассчитывает новые scores;
4. ranking базовой модели и Cross-Encoder объединяются;
5. выбираются итоговые Top-50.

На этом этапе отдельно считаются Recall@50 и Candidate Recall, а также метрики для regular и special запросов.

In [30]:
ADAPTER_HASH = {p.name: file_sha256(p) for p in sorted(ADAPTER_DIR.iterdir()) if p.is_file()}
assert "adapter_config.json" in ADAPTER_HASH
CE_SCORE_BASE = {
    "adapter": ADAPTER_HASH, "dtype": str(CE_DTYPE),
    "base_sha256": BASE_MODEL_HASH, "tokenizer_sha256": TOKENIZER_HASH,
    "max_length": CFG["ce_max_length"],
    "document_text": document_meta.get("identity", {}).get("text_hash"),
    "cfg": CFG_ID,
}
ce_model.eval()

def score_candidate_parts(candidate_dir, frame, tag, rows_per_part=8192):
    key_col = "query_id" if "query_id" in frame else "context_id"
    text_map = frame.assign(_key=frame[key_col].astype(str)).set_index("_key").query_text.astype(str).to_dict()
    candidates = read_candidate_parts(candidate_dir)
    assert candidates.key.nunique() == len(frame)
    pair_columns = ["key", "item_row", "base_score", "base_rank", "special_used"]
    identity = {**CE_SCORE_BASE, "version": "final_score_v1",
                "candidate_content": frame_digest(candidates, pair_columns),
                "text_map": frame_digest(frame, [key_col, "query_text"]), "n_rows": len(candidates)}
    folder = persist_manifest(V5_DIR / f"scores_{tag}", identity)
    for start in tqdm(range(0, len(candidates), rows_per_part), desc=f"fine-tuned scores {tag}"):
        expected = candidates.iloc[start:start + rows_per_part].reset_index(drop=True)
        path = folder / f"part_{start:09d}.parquet"
        if path.exists():
            old = pd.read_parquet(path)
            if (len(old) != len(expected) or
                    frame_digest(old, pair_columns) != frame_digest(expected, pair_columns) or
                    not np.isfinite(old.ce_score.to_numpy()).all()):
                raise RuntimeError(f"Повреждены сохранённые CE scores: {path}")
            continue
        scores = np.empty(len(expected), dtype="float32")
        local = 0
        bs = CFG["ce_infer_batch"]
        while local < len(expected):
            limit = min(local + bs, len(expected))
            batch = expected.iloc[local:limit]
            qtexts = [text_map[str(k)] for k in batch.key]
            dtexts = items.text.iloc[batch.item_row.to_numpy("int32")].astype(str).tolist()
            enc, logits = None, None
            try:
                enc = tokenizer(qtexts, dtexts, padding=True, truncation=True,
                                max_length=CFG["ce_max_length"], return_tensors="pt")
                enc = {k: v.to("cuda") for k, v in enc.items()}
                with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=CE_DTYPE):
                    logits = ce_model(**enc).logits.reshape(-1)
                assert logits.numel() == len(batch)
                scores[local:limit] = logits.float().cpu().numpy()
                local = limit
            except torch.cuda.OutOfMemoryError:
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                print(f"{tag}: CE batch → {bs} after CUDA OOM")
            finally:
                del enc, logits
            if local < limit:
                gc.collect()
                torch.cuda.empty_cache()
        assert np.isfinite(scores).all()
        expected["ce_score"] = scores
        atomic_parquet(expected, path)
    del candidates
    gc.collect()
    return folder

def read_scored_parts(folder):
    files = sorted(Path(folder).glob("part_*.parquet"))
    if not files:
        raise RuntimeError(f"Нет CE scores: {folder}")
    scored = pd.concat((pd.read_parquet(p) for p in files), ignore_index=True)
    scored["key"] = scored.key.astype(str)
    assert np.isfinite(scored.ce_score.to_numpy()).all()
    assert not scored.duplicated(["key", "item_row"]).any()
    return scored

In [31]:
INNER_SCORE_DIR = score_candidate_parts(INNER_CAND_DIR, v5_valid_q, "inner")

fine-tuned scores inner: 100%|██████████| 52/52 [20:46<00:00, 23.97s/it]


In [32]:
def evaluate_fixed(scored, frame):
    expected_keys = set(frame.context_id.astype(str))
    assert set(scored.key) == expected_keys
    rows = []
    for key, group in tqdm(
        scored.groupby("key", sort=False), total=len(expected_keys), desc="Recall@50"
    ):
        group = group.sort_values("base_rank")
        ids = group.item_row.to_numpy("int32")
        base = group.base_score.to_numpy("float32")
        ce = group.ce_score.to_numpy("float32")
        assert len(ids) >= 50 and len(np.unique(ids)) == len(ids)
        fused = (
            (1 - CFG["gamma"]) * rank_reciprocal(base, ids)
            + CFG["gamma"] * rank_reciprocal(ce, ids)
        )
        ranking = order_by(ids, fused)[:50]
        p = positive_rows[str(key)]
        rows.append({
            "context_id": str(key),
            "special_used": bool(group.special_used.iloc[0]),
            "n_positive": len(p),
            **candidate_metrics(ids, p),
            **ranking_metrics(ranking, p),
        })
    return pd.DataFrame(rows)


def metric_summary(table, split):
    cols = ["Recall50", "CandidateRecall", "Oracle50", "MRR50"]
    parts = []
    for segment, mask in [
        ("overall", np.ones(len(table), dtype=bool)),
        ("regular", ~table.special_used.to_numpy()),
        ("special", table.special_used.to_numpy()),
    ]:
        row = table.loc[mask, cols].mean().to_dict()
        row.update({"split": split, "segment": segment, "contexts": int(mask.sum())})
        parts.append(row)
    return pd.DataFrame(parts)[["split", "segment", "contexts", *cols]]


inner_scored = read_scored_parts(INNER_SCORE_DIR)
inner_eval = evaluate_fixed(inner_scored, v5_valid_q)
inner_summary = metric_summary(inner_eval, "Validation")
display(inner_summary.round(6))
atomic_parquet(inner_eval, V5_DIR / "validation_inner_metrics.parquet")
del inner_scored
gc.collect()

Recall@50: 100%|██████████| 2500/2500 [00:00<00:00, 3052.11it/s]


,split,segment,contexts,Recall50,CandidateRecall,Oracle50,MRR50
0,Validation,overall,2500,0.819116,0.879810,0.879810,0.320623
1,Validation,regular,2164,0.839952,0.889798,0.889798,0.338885
2,Validation,special,336,0.684921,0.815476,0.815476,0.203006


0

## 15. Holdout

После Validation те же зафиксированные параметры применяются к отдельному Holdout. Этот блок не переобучает модель и нужен только для дополнительной проверки устойчивости результата.

In [33]:
HOLDOUT_CAND_DIR = build_adaptive_candidates(
    holdout_5000, "holdout", "full", regular_eval_ranker, special_eval_ranker, "dev"
)

candidates holdout: 100%|██████████| 40/40 [04:14<00:00,  6.36s/it]


In [34]:
HOLDOUT_SCORE_DIR = score_candidate_parts(HOLDOUT_CAND_DIR, holdout_5000, "holdout")
holdout_scored = read_scored_parts(HOLDOUT_SCORE_DIR)
holdout_eval = evaluate_fixed(holdout_scored, holdout_5000)
holdout_summary = metric_summary(holdout_eval, "Holdout")
display(holdout_summary.round(6))
atomic_parquet(holdout_eval, V5_DIR / "validation_holdout_metrics.parquet")
del holdout_scored
gc.collect()

Recall@50: 100%|██████████| 5000/5000 [00:01<00:00, 2776.47it/s]


,split,segment,contexts,Recall50,CandidateRecall,Oracle50,MRR50
0,Holdout,overall,5000,0.821988,0.881371,0.881371,0.324870
1,Holdout,regular,4199,0.848382,0.890436,0.890436,0.340679
2,Holdout,special,801,0.683625,0.833854,0.833854,0.241992


0

In [35]:
evaluation_summary = pd.concat([inner_summary, holdout_summary], ignore_index=True)
display(evaluation_summary.round(6))

,split,segment,contexts,Recall50,CandidateRecall,Oracle50,MRR50
0,Validation,overall,2500,0.819116,0.879810,0.879810,0.320623
1,Validation,regular,2164,0.839952,0.889798,0.889798,0.338885
2,Validation,special,336,0.684921,0.815476,0.815476,0.203006
3,Holdout,overall,5000,0.821988,0.881371,0.881371,0.324870
4,Holdout,regular,4199,0.848382,0.890436,0.890436,0.340679
5,Holdout,special,801,0.683625,0.833854,0.833854,0.241992


## 16. Фиксация финальной модели и Benchmark

После Validation и Holdout конфигурация считается зафиксированной.

In [36]:
FREEZE_FILE = V5_DIR / "model_freeze.json"
freeze_identity = {
    "train_id": TRAIN_ID,
    "adapter": ADAPTER_HASH,
    "cfg_id": CFG_ID,
    "inner_contexts": len(inner_eval),
    "holdout_contexts": len(holdout_eval),
    "regular_K": CFG["regular_K"],
    "special_K": CFG["special_K"],
    "gamma": CFG["gamma"],
}
if FREEZE_FILE.exists():
    assert json.loads(FREEZE_FILE.read_text()) == freeze_identity
else:
    atomic_json(FREEZE_FILE, freeze_identity)

benchmark_frame = queries.copy()
BENCH_CAND_DIR = build_adaptive_candidates(
    benchmark_frame,
    "benchmark",
    "benchmark",
    final_regular_ranker,
    final_special_ranker,
    "full",
)
assert FREEZE_FILE.exists()

candidates benchmark: 100%|██████████| 20/20 [01:25<00:00,  4.27s/it]


## 17. Submission

Делаем предсказания для получения итоговой метрики на сайте

In [37]:
BENCH_SCORE_DIR = score_candidate_parts(BENCH_CAND_DIR, benchmark_frame, "benchmark")
benchmark_scored = read_scored_parts(BENCH_SCORE_DIR)
expected_benchmark_ids = benchmark_frame.query_id.astype(str).tolist()
assert len(expected_benchmark_ids) == 2452
assert set(benchmark_scored.key) == set(expected_benchmark_ids)
assert benchmark_scored.item_row.between(0, BENCH_N - 1).all()
benchmark_answers = {}
for key, group in tqdm(benchmark_scored.groupby("key", sort=False), total=len(benchmark_frame), desc="submission"):
    group = group.sort_values("base_rank")
    ids = group.item_row.to_numpy("int32")
    assert len(ids) >= 50 and len(ids) == len(np.unique(ids))
    base = group.base_score.to_numpy("float32")
    ce = group.ce_score.to_numpy("float32")
    fused = (1 - CFG["gamma"]) * rank_reciprocal(base, ids) + CFG["gamma"] * rank_reciprocal(ce, ids)
    ranked = order_by(ids, fused)[:50]
    assert len(ranked) == len(np.unique(ranked)) == 50 and (ranked < BENCH_N).all()
    benchmark_answers[str(key)] = " ".join(items.item_id.iloc[ranked].astype(str))
submission = pd.DataFrame({"query_id": expected_benchmark_ids})
submission["answer"] = submission.query_id.map(benchmark_answers)
assert len(submission) == len(benchmark_frame) and submission.query_id.is_unique
assert submission.answer.notna().all()
assert submission.query_id.tolist() == expected_benchmark_ids
allowed_ids = set(items.iloc[:BENCH_N].item_id.astype(str))
tokens = submission.answer.str.split()
assert tokens.map(len).eq(50).all()
assert tokens.map(lambda x: len(set(x))).eq(50).all()
assert all(item in allowed_ids for group in tokens for item in group)
SUBMISSION = V5_DIR / "submission.csv"
tmp = SUBMISSION.with_suffix(".tmp.csv")
submission.to_csv(tmp, index=False)
os.replace(tmp, SUBMISSION)
submission_check = pd.DataFrame({
    "queries": [len(submission)],
    "items_per_query": [int(tokens.map(len).iloc[0])],
    "all_unique_within_query": [bool(tokens.map(lambda x: len(set(x))).eq(50).all())],
})
display(submission_check)
del benchmark_scored
gc.collect()

submission: 100%|██████████| 2452/2452 [00:01<00:00, 2207.39it/s]


,queries,items_per_query,all_unique_within_query
0,2452,50,True


0

## 18. Итог


После успешного запуска сохраняются:

- обученный LoRA adapter;
- параметры финальной модели;
- метрики Validation;
- метрики Holdout;
- итоговый `submission.csv`.

Итоговая метрика на локальных данных:

In [41]:
final_local_recall50 = float(holdout_eval["Recall50"].mean())
print(f"FINAL LOCAL Recall@50: {final_local_recall50:.6f}")


FINAL LOCAL Recall@50: 0.821988


In [38]:
assert SUBMISSION.is_file() and FREEZE_FILE.is_file() and TRAINING_DONE.is_file()
assert (V5_DIR / "training_log.csv").is_file()
assert (ADAPTER_DIR / "adapter_config.json").is_file()
assert list(CKPT_DIR.glob("step_*/commit.json"))
assert INNER_SCORE_DIR.is_dir() and HOLDOUT_SCORE_DIR.is_dir() and BENCH_SCORE_DIR.is_dir()
assert pd.read_csv(SUBMISSION).shape[0] == 2452
assert len(submission) == 2452
print("Pipeline completed. Submission:", SUBMISSION.resolve())

Pipeline completed. Submission: /home/ubuntu/avito/artifacts/final_v51_clean/run_781f7097a8530584/submission.csv
